# 🧪 SegFormer 2-Head Coral Segmentation

Trains **SegFormer-B2** on your coral CSV + (optionally) the Coralscapes dataset as a regularizer. One model, two prediction heads — each head keeps its own class list so nothing is remapped.

---

## 🚀 Quick start (TL;DR)

1. In **Cell 2** edit the **EDIT-THIS block** at the top (STAGE, KEEP_TOP_N, NUM_EPOCHS, DROP_CLASSES).
2. Pick a GPU (see table below), then `Runtime → Run all`.
3. When finished, you get:
   - `{EXPERIMENT_DIR}/bundle.pt` → auto-downloads to your browser. Drop it into `webapp/segformer_predict_app.py` (Streamlit) to validate.
   - `{EXPERIMENT_DIR}/README.md` → human-readable settings + metrics overview.
   - `{EXPERIMENT_DIR}/summary.json` → machine-readable version.
4. Run **Cell 12** to A/B compare this run against a previous one.

---

## 🖥️ Which Colab GPU to pick

Rough numbers for ~4400 images, 512 px input, batch 8, MiT-B2:

| GPU | VRAM | Cost (units/h) | `points` 20ep | `sam_masks` 30ep | When to use |
|---|---|---|---|---|---|
| **T4** (free) | 16 GB | 0 | ~40 min | ~90 min | Default for a first run. Free tier. Slow but works. |
| **L4** | 22 GB | ~4 | ~20 min | ~45 min | **Sweet spot.** 2× faster than T4 for ~4× the cost. Pick this for most real runs. |
| **A100** | 40 GB | ~12 | ~10 min | ~22 min | Only if you need speed (e.g. comparing many settings). 3× faster than L4 for 3× the cost — pure speed trade. |
| **V100** | 16 GB | ~5 | ~25 min | ~55 min | Older card, roughly L4-equivalent. Only if L4 isn't offered in your region. |

**Decision rule:**
- Just exploring / first time / don't care about speed → **T4**.
- Iterating on real settings → **L4**. Best value.
- Running 10+ experiments in a session → **A100**. Time beats cost.

**Memory note:** B2 at 512 px batch 8 fits in ~6 GB VRAM, so any of these is safe. If you bump `MODEL_NAME` to `mit-b3` or `mit-b5`, drop batch size to 4 or 2 on T4; L4 and A100 are fine up to B5.

**SAM batch export** (`batch_export_sam.ipynb`) scales with image count: ~4 h on T4, ~45 min on L4, ~30 min on A100 for 4400 images. Do this on L4 or A100 — it's a one-time cost you amortize across every SegFormer experiment afterwards.

---

## 📁 Where everything saves

Every run creates its own folder on Drive, named by the settings it used — **no more overwriting the previous model**:

```
/content/drive/MyDrive/coral_training/segformer_runs/
├── experiments/
│   ├── points_top35_clean_20ep/            ← one experiment per folder
│   │   ├── best.pt          # highest-val-mIoU checkpoint (for resuming)
│   │   ├── final.pt         # last-epoch checkpoint
│   │   ├── bundle.pt        # self-contained: weights + classes + config (use this in the app)
│   │   ├── summary.json     # machine-readable run record
│   │   ├── README.md        # human-readable settings + metrics
│   │   └── history.json     # per-epoch loss / mIoU
│   ├── sam_masks_top35_clean_30ep/
│   └── ...
└── checkpoints/              # legacy flat files from older runs (kept for resume)
```

The folder name is auto-generated from **STAGE**, **KEEP_TOP_N_CLASSES**, whether `DROP_CLASSES` is non-empty, and **NUM_EPOCHS** so you always know what each experiment was. Override `EXPERIMENT_NAME` in Cell 2 if you want a custom name.

---

## 🧭 Workflow — 3 stages (run in order)

| Stage | Labels used | Typical mIoU | When to run |
|---|---|---|---|
| `points` | Your CSV points (1 labeled pixel per point, rest ignored) | 0.05-0.15 | **Always first.** Sparse but trustworthy. Validates the pipeline end-to-end. Patch accuracy is the real metric here, not mIoU. |
| `sam_masks` | `sam_coco.json` from `batch_export_sam.ipynb` | 0.25-0.45 | After SAM preview looks clean. **Big quality jump** because labels are dense. Resumes from `points` automatically. |
| `pseudo` | Stage 2 predictions + SAM masks combined, filtered by confidence | 0.30-0.50 | Optional final refinement. Only worth it if `sam_masks` plateaued. |

---

## 🔀 How to compare two runs

Open **Cell 12**. Set the two experiment names you want to compare (or leave them empty for auto-pick: the most recent `points` vs the most recent `sam_masks`).

The cell:
1. Prints all settings side-by-side (epochs, classes, SAM params, etc.).
2. Re-runs validation with both weights on the **same val set** and reports Δ mIoU_B.
3. Renders side-by-side predictions on the same test image.
4. Verdicts the result: `✅ B wins`, `❌ B regressed`, or `➖ tie`.

---

## ⚠️ Important design rules

- `ignore_index=255` for uncovered pixels → cross-entropy skips them. SegFormer trains fine on sparse labels.
- Points labeled as `"Unknown"` / sentinels are **dropped from training** (become IGNORE). Otherwise the model learns a trash class that absorbs anything it's unsure about.
- When `KEEP_TOP_N_CLASSES` is set, rare-class pixels also become IGNORE — the model isn't taught to misclassify them, they just don't contribute to the loss.
- **COCO / CSV are the source of truth.** Cell 5 re-renders PNG masks every run with caching.
- Test images `TEST_IMAGES` are the same across this notebook, `sam_viewer.py`, and `batch_export_sam.ipynb` → side-by-side visual comparison across tools.

---

## ✅ After training — what to check

1. Does **Cell 9**'s top/bottom-10 class table look sane? (Common classes high, rare classes low.)
2. Does **Cell 10**'s visualization look like the image content? Boundaries blobby for `points`, sharper for `sam_masks`.
3. Did **Cell 11** print `✅ converged` or `📈 still improving`? If improving, bump `NUM_EPOCHS` and re-run.
4. Does **Cell 12** say `B wins` when comparing against the previous stage? If not, the SAM export may need tuning (`AMG_PRED_IOU_THRESH`, `AMG_STABILITY_THRESH` in `batch_export_sam.ipynb` Cell 2).

---

## 🧰 Recommended experiments to run (in order)

1. `points` + `KEEP_TOP_N_CLASSES = 35` + drop Unknown — clean baseline.
2. `sam_masks` + same class filter — should beat points significantly.
3. `sam_masks` + `KEEP_TOP_N_CLASSES = None` (all 87 classes) — if you need rare-species coverage; expect lower mIoU but better tail.
4. `pseudo` — only if #2 converged well.

Each run lives in its own folder, so you can always roll back to a better one.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                     ⚡  QUICK SETTINGS  ⚡                              ║
# ║  Edit here — these override Cell 2 defaults. Run this cell first.       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Training flow ─────────────────────────────────────────────────────────────
# "2"   = CSV point annotations only  (fast, ~20 epochs)
# "4"   = Gold COCO polygons only     (best quality, ~15 epochs)
# "24"  = Points → Gold               (recommended when you have both)
# "124" = Pretrain on points → fine-tune on Gold  (full pipeline)
TRAINING_FLOW = "124"

# ── Drive folder (everything else is relative to this) ───────────────────────
DRIVE_DIR = '/content/drive/MyDrive/coral_training'

# ── Input files ───────────────────────────────────────────────────────────────
YOUR_CSV_PATH       = f'{DRIVE_DIR}/annotations_coralnet.csv'   # CoralNet point CSV
YOUR_SAM_COCO_PATH  = f'{DRIVE_DIR}/sam_coco.json'              # SAM-generated masks (stage 3)
YOUR_GOLD_COCO_PATH = f'{DRIVE_DIR}/gold_coco.json'             # Roboflow expert polygons (stage 4)
YOUR_IMAGES_DRIVE   = f'{DRIVE_DIR}/images'                     # flat or nested image folder

# ── Re-render masks? ──────────────────────────────────────────────────────────
# Set True after uploading new images/annotations, or after changing POINT_RADIUS.
# False = use cached masks (faster restarts).
FORCE_RERENDER = True   # ← rebuilds masks (set False after first run to use cache)

# ── Model & training ──────────────────────────────────────────────────────────
MODEL_NAME  = "nvidia/mit-b2"   # mit-b0 (fast) → mit-b5 (best, needs more VRAM)
BATCH_SIZE  = 8                 # reduce to 4 if out-of-memory
INPUT_SIZE  = 512               # 512 is standard; 640 for higher detail (needs more VRAM)
USE_AMP     = True              # mixed-precision — keep True on T4/A100

# ── Class filtering ───────────────────────────────────────────────────────────
KEEP_TOP_N_CLASSES = 35         # None = keep all; 35 = drop rare classes for cleaner training
USE_LABEL_MERGE    = True       # Merge 94 CoralNet labels → ~49 broader classes

# ── Resume ────────────────────────────────────────────────────────────────────
# None = auto-resume from previous stage if checkpoint exists, else start fresh
# "experiment_name" = resume from a specific named run
RESUME_FROM_EXPERIMENT = None


# ── Loss for stages 3/4 (dense polygon supervision) ───────────────────────────
# "ce"       = cross-entropy (original; may over-smooth predictions to maximize mIoU)
# "dice_ce"  = Dice + CE combo  ← RECOMMENDED — prevents over-prediction shortcut,
#              keeps coral colony edges sharper. Slight mIoU drop possible.
# "focal_ce" = focal cross-entropy (focus on hard pixels / boundaries)
# "dice"     = Dice only
STAGE4_LOSS = "dice_ce"

print("✅ Quick settings loaded — these override Cell 2 defaults.")
print(f"   TRAINING_FLOW={TRAINING_FLOW!r}  BATCH_SIZE={BATCH_SIZE}  INPUT_SIZE={INPUT_SIZE}  FORCE_RERENDER={FORCE_RERENDER}")
print(f"   DRIVE_DIR={DRIVE_DIR!r}")


In [ ]:
# ===========================================================================
# CELL 1: Install dependencies, mount Drive, import everything
# ===========================================================================
# Only kills the kernel if a real numpy binary incompatibility is detected.
# Normal case: installs packages, imports cleanly, no restart needed.
# The force-reinstall of numpy at the end of install prevents downgrades from
# transitive deps (pycocotools etc.).
# ===========================================================================

import os, subprocess, sys

_MARKER = '/content/.coral_deps_installed'

if not os.path.exists(_MARKER):
    print('Installing dependencies (one-time per VM)...')
    _pkgs = [
        "numpy>=2,<3",
        "transformers>=4.44,<5",
        "accelerate>=0.33",
        "albumentations>=1.4.18",
        "datasets>=2.21",
        "pycocotools",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs], check=True)
    # Force numpy back to 2.x in case any transitive dep pulled in 1.x
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--force-reinstall", "--no-deps", "numpy>=2,<3"],
        check=True,
    )
    with open(_MARKER, 'w') as f:
        f.write('ok')
    print('✅ Dependencies installed.')


# --- Import everything; only kill kernel if a real binary mismatch occurs ---
def _kill_with_message(reason):
    print(f'⚠️ {reason}')
    print('   Restarting kernel automatically — click ▶ on this cell again to continue.')
    os.kill(os.getpid(), 9)

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import json, random, time, gc
    from collections import Counter, defaultdict

    import numpy as np
    import pandas as pd
    import cv2

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.amp import autocast, GradScaler
    _autocast = lambda enabled: autocast("cuda", enabled=enabled)

    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    from transformers import (
        SegformerConfig,
        SegformerDecodeHead,
        SegformerForSemanticSegmentation,
    )

except ValueError as e:
    if 'dtype size changed' in str(e) or 'binary incompatibility' in str(e):
        _kill_with_message('numpy binary incompatibility — kernel has stale numpy in memory')
    raise
except ImportError as e:
    if 'numpy' in str(e).lower():
        _kill_with_message(f'Import error hinting at numpy version issue: {e}')
    raise

# --- GPU speedups (quality-safe) ---
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print(f'numpy {np.__version__}, torch {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('cuDNN benchmark + TF32 enabled')
else:
    print('⚠️ No GPU detected. Switch runtime to GPU (L4 recommended).')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIG — edit the block at the top, everything below auto-derives
# ═══════════════════════════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  ✏️  EDIT THIS OFTEN — the knobs you actually change between runs       │
# └─────────────────────────────────────────────────────────────────────────┘

# --- Which stage + how long -----------------------------------------------


# ═══════════════════════════════════════════════════════════════════════════
# LABEL MERGE MAP — 2025-07 taxonomy simplification (94 → 49 classes)
# Embedded inline so this notebook is fully self-contained.
# Source of truth: colab/label_mapping.py in the project repo.
# ═══════════════════════════════════════════════════════════════════════════
_LABEL_MERGE_MAP = {
    # → Rock
    'TA': 'Rock', 'HS_AR': 'Rock', 'CCA': 'Rock',
    'Dead (ex)': 'Rock', 'Biofilm ex': 'Rock',
    # → SP
    'Didae': 'SP', 'Tun': 'SP',
    # → GA
    'Hal': 'GA', 'Fila (ex)': 'GA', 'Val': 'GA', 'Cau': 'GA',
    # → HC  (rare genera, each < 400 annotations)
    'Turb-HC': 'HC', 'Myc': 'HC', 'Echphy': 'HC', 'Cyph': 'HC',
    'Diplo': 'HC', 'Ser': 'HC', 'Lepta': 'HC', 'Mer': 'HC',
    'Oul': 'HC', 'Leptos': 'HC', 'Leptor': 'HC', 'Dun': 'HC',
    'Plero': 'HC', 'Bla': 'HC', 'Oxy': 'HC', 'Para': 'HC',
    'Pod': 'HC', 'Pachy': 'HC', 'Alv': 'HC', 'Sym': 'HC', 'Psa': 'HC',
    # → Frame / Xe / Other
    'Urchins ex': 'Frame', 'Tubmus': 'Xe',
    'Zoan': 'Other', 'Bivalve': 'Other', 'Bryo': 'Other',
}
_EXCLUDED_LABELS = {
    'Unknown', 'Unkn', 'Unk', 'MA', 'BA',
    '?', 'NA', 'nan', 'None', '', 'Off',
}

def apply_label_merge(label):
    """Map a CoralNet label to its merged class, or None if excluded."""
    if label in _EXCLUDED_LABELS:
        return None
    return _LABEL_MERGE_MAP.get(label, label)

def merge_dataframe_labels(df, label_col='Label'):
    """Apply merge map to df[label_col]; drops excluded rows. Returns (df, n_dropped)."""
    df = df.copy()
    df[label_col] = df[label_col].map(apply_label_merge)
    before = len(df)
    df = df[df[label_col].notna()].reset_index(drop=True)
    return df, before - len(df)

USE_LABEL_MERGE = globals().get("USE_LABEL_MERGE", True)

# ── Training pipeline flow ────────────────────────────────────────────────
# Configure which stages to run and chain them automatically.
#
#   1  =  Coralscapes warm-start   (pretrained weights, no extra training)
#   2  =  CoralNet patches         (point annotations → small disks)
#   3  =  SAM mask propagation     (from batch_export_sam.ipynb)
#   4  =  Gold-truth masks         (Roboflow professional COCO annotations)
#   5  =  Targeted refinement      (extra Roboflow annotations of weak classes;
#                                   resumes from your BEST 1-4 experiment)
#
# Examples:
#   "12"   — patches fine-tune only  (fastest, good baseline)
#   "124"  — patches → gold          (skip SAM if biologist data is available)
#   "1234" — full pipeline           (patches → SAM → gold, best final quality)
#   "123"  — patches → SAM only      (if no gold masks yet)
#   "5"    — targeted refinement only (run AFTER seeing Cell 9 confusion matrix;
#                                       auto-resumes from highest-mIoU prior run)
#
# Each fine-tuning stage automatically loads the best checkpoint from the
# previous stage. Coralscapes warm-start (1) is always the foundation.
TRAINING_FLOW = globals().get("TRAINING_FLOW", "12")  # set in Quick Settings

# Per-stage epochs and learning rate (edit if you want finer control)
STAGE_CONFIGS = {
    "2": {"label": "patches",    "stage": "points",     "epochs": 20,  "lr": 6e-5},
    "3": {"label": "sam_masks",  "stage": "sam_masks",  "epochs": 12,  "lr": 2e-5},
    "4": {"label": "gold_masks", "stage": "gold_masks", "epochs": 8,   "lr": 3e-6},  # gentle: preserve stage 2 texture detail (was 30ep@1e-5 → over-smoothed edges)
    # Stage 5: targeted refinement on additional Roboflow annotations
    # targeting classes that performed poorly in Cell 9's confusion matrix.
    # Lower LR (5e-6) because the prior model is already close to optimal —
    # we want gentle nudging on the new examples, not catastrophic forgetting.
    "5": {"label": "targeted",   "stage": "targeted",   "epochs": 10,  "lr": 5e-6},
}
# Convenience: first training stage sets the global STAGE used by downstream cells
_flow_stages = [s for s in TRAINING_FLOW if s != "1"]
if _flow_stages:
    STAGE      = STAGE_CONFIGS[_flow_stages[0]]["stage"]
    NUM_EPOCHS = STAGE_CONFIGS[_flow_stages[0]]["epochs"]
    LR         = STAGE_CONFIGS[_flow_stages[0]]["lr"]
else:
    STAGE = "points"

# ── Note on USE_LABEL_MERGE ──────────────────────────────────────────────
# Toggle is at the TOP of this cell (inside the label merge map block).
# Set USE_LABEL_MERGE = True/False there — merges 94 CoralNet labels → ~49.
# ALWAYS retrain after changing — post-hoc merging is ~5-10 mIoU worse.


# --- Class filtering -------------------------------------------------------
KEEP_TOP_N_CLASSES = globals().get('KEEP_TOP_N_CLASSES', 35)  # None = keep all

DROP_CLASSES = [                   # Labels to IGNORE in training (not predicted as a class)
    "Unknown", "Unkn", "Unk",      # Sentinel annotator labels — see notebook intro for why this matters
    "Off", "?", "NA", "nan", "None", "",
]

# --- Coralscapes regularizer (on/off) -------------------------------------
USE_PRETRAINED_CORAL    = True     # Start from HF EPFL-ECEO/coralscapes weights. Keep True — free quality.
USE_CORALSCAPES_DATASET = False    # Also TRAIN on Coralscapes images. Slower, marginal gain. Leave False unless experimenting.

# --- Experiment naming + resume (optional overrides) ----------------------
# Leave None for auto-names like "sam_masks_top35_clean_10ep" (one folder per run, no overwrites).
# Set a custom name if LR (or another non-name-encoded setting) differs and you want clarity:
#   EXPERIMENT_NAME = "sam_masks_top35_lr2e5_10ep"
EXPERIMENT_NAME        = None

# Leave None for auto-resume from previous stage  (points ← nothing,
# sam_masks ← points, pseudo ← sam_masks).  Set a folder name under
# experiments/ to resume from a specific run.
RESUME_FROM_EXPERIMENT = globals().get("RESUME_FROM_EXPERIMENT", None)

# --- Skip the 3-5 min Drive → SSD image sync ------------------------------
#   False (default) : sync images to fast SSD. REQUIRED for training — otherwise
#                     each epoch is ~10× slower because Drive is a network mount.
#   True            : read images directly from Drive. FINE for any run that
#                     doesn't train from scratch, i.e.:
#                       - evaluation only (Cell 9 + 11)
#                       - comparing models (Cell 12)
#                       - re-bundling an existing checkpoint (Cell 11 only)
#                       - resuming training on a VM where images are already synced
SKIP_IMAGE_SYNC = globals().get("SKIP_IMAGE_SYNC", False)


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  🔧 RARELY CHANGED — architecture, paths, worker counts                 │
# └─────────────────────────────────────────────────────────────────────────┘

# --- Paths on Drive ---
DRIVE_DIR          = globals().get('DRIVE_DIR', '/content/drive/MyDrive/coral_training')
YOUR_IMAGES_DRIVE  = f'{DRIVE_DIR}/images'
YOUR_CSV_PATH      = f'{DRIVE_DIR}/annotations_coralnet.csv'
YOUR_SAM_COCO_PATH  = f'{DRIVE_DIR}/sam_coco.json'
YOUR_GOLD_COCO_PATH = f'{DRIVE_DIR}/gold_coco.json'    # raw Roboflow export — cleaned automatically in Cell 5
YOUR_TARGETED_COCO_PATH = f'{DRIVE_DIR}/targeted_coco.json'  # stage 5: extra annotations for weak classes (separate Roboflow project)

OUTPUT_DIR         = f'{DRIVE_DIR}/segformer_runs'
EXPERIMENTS_DIR    = f'{OUTPUT_DIR}/experiments'
CKPT_DIR           = f'{OUTPUT_DIR}/checkpoints'     # legacy flat location (kept for back-compat)

# --- Derived flag for Coralscapes joint training ---
HAS_CORALSCAPES = USE_CORALSCAPES_DATASET

CORALSCAPES_HF_REPO     = "EPFL-ECEO/coralscapes"
CORALSCAPES_CACHE_DIR   = "/content/coralscapes_cache"
CORALSCAPES_IMAGES      = f'{CORALSCAPES_CACHE_DIR}/images'
CORALSCAPES_MASKS       = f'{CORALSCAPES_CACHE_DIR}/masks'
CORALSCAPES_CLASSES_TXT = f'{CORALSCAPES_CACHE_DIR}/classes.txt'

# --- Model architecture ---
MODEL_NAME   = globals().get("MODEL_NAME", "nvidia/mit-b2")
INPUT_SIZE   = globals().get("INPUT_SIZE", 512)
IGNORE_INDEX = 255

# --- Training hyperparameters (LR lives in the EDIT THIS OFTEN block above) ---
BATCH_SIZE   = globals().get("BATCH_SIZE", 8)
WEIGHT_DECAY = 1e-4
SEED         = 42
USE_AMP      = globals().get("USE_AMP", True)
STAGE4_LOSS  = globals().get("STAGE4_LOSS", "dice_ce")  # "ce" to revert to original

# --- DataLoader ---
NUM_WORKERS        = 4
PREFETCH_FACTOR    = 4
PERSISTENT_WORKERS = True

# --- Joint training (Coralscapes sampling probability, decays over epochs) ---
CORALSCAPES_WEIGHT_START = 0.5
CORALSCAPES_WEIGHT_END   = 0.2

# --- Test images (shared with sam_viewer.py + batch_export_sam.ipynb) ---
TEST_IMAGES = [
    "DSCN2121.jpg", "G0081188.jpg", "trainingJ10.jpg",
]
N_TEST_IMAGES = 10

# --- Local cache dirs (fast SSD) ---
MASKS_CACHE_DIR   = '/content/masks_cache'
LOCAL_IMAGES_DIR  = '/content/images_local'

# ═══════════════════════════════════════════════════════════════════════════
# Derived values — auto-computed from the knobs above. Don't edit below.
# ═══════════════════════════════════════════════════════════════════════════

import subprocess

# --- Resume-from-previous-stage logic ---
_AUTO_RESUME_FROM_STAGE = {
    "points":     None,
    "sam_masks":  "points",
    "gold_masks": "sam_masks",
    "pseudo":     "sam_masks",
    "targeted":   "__BEST__",   # stage 5: auto-resolve to highest-mIoU prior experiment
}.get(STAGE)  # .get() avoids KeyError for unknown stage names

RESUME_FROM_STAGE = _AUTO_RESUME_FROM_STAGE  # back-compat alias used in Cell 7

# --- Auto-derive EXPERIMENT_NAME if not provided ---
def _auto_experiment_name(stage, keep_top_n, drop_classes, n_epochs):
    parts = [stage]
    parts.append(f"top{keep_top_n}" if keep_top_n else "all")
    if drop_classes:
        parts.append("clean")          # indicates DROP_CLASSES is non-empty
    parts.append(f"{n_epochs}ep")
    return "_".join(parts)

if not EXPERIMENT_NAME:
    EXPERIMENT_NAME = _auto_experiment_name(STAGE, KEEP_TOP_N_CLASSES, DROP_CLASSES, NUM_EPOCHS)

EXPERIMENT_DIR = f'{EXPERIMENTS_DIR}/{EXPERIMENT_NAME}'

# --- Auto-resolve which experiment to resume from ---
def _pick_best_experiment(experiments_root):
    """Stage 5 helper: scan all experiment folders, return (name, miou) of the
    highest-scoring one. Looks at summary.json then best.pt. Lets stage 5
    resume from the strongest prior run automatically — you don't have to
    remember which combo of stages 1-4 won."""
    if not os.path.isdir(experiments_root):
        return None, None
    best = (None, -1.0)
    for name in os.listdir(experiments_root):
        d = os.path.join(experiments_root, name)
        if not os.path.isdir(d):
            continue
        miou = None
        sj = os.path.join(d, 'summary.json')
        if os.path.exists(sj):
            try:
                with open(sj) as f:
                    data = json.load(f)
                miou = (data.get('best_mIoU') or data.get('val_miou_B')
                        or (data.get('summary') or {}).get('mIoU'))
            except Exception:
                miou = None
        if miou is None:
            bp = os.path.join(d, 'best.pt')
            if os.path.exists(bp):
                try:
                    sd = torch.load(bp, map_location='cpu', weights_only=False)
                    miou = float(sd.get('miou', -1))
                except Exception:
                    miou = None
        if miou is not None and miou > best[1]:
            best = (name, float(miou))
    return best if best[0] else (None, None)


def _resolve_resume_path():
    if RESUME_FROM_EXPERIMENT:
        p = os.path.join(EXPERIMENTS_DIR, RESUME_FROM_EXPERIMENT, 'best.pt')
        return p if os.path.exists(p) else None
    # Stage 5 (targeted): pick the best of all prior experiments
    if _AUTO_RESUME_FROM_STAGE == "__BEST__":
        _name, _miou = _pick_best_experiment(EXPERIMENTS_DIR)
        if _name:
            print(f'🏆 targeted stage: auto-picked best prior experiment '
                  f'"{_name}" (mIoU={_miou:.4f})')
            return os.path.join(EXPERIMENTS_DIR, _name, 'best.pt')
        print('⚠️  targeted stage: no prior experiments found in '
              f'{EXPERIMENTS_DIR}. Run stages 1-4 first, or set '
              'RESUME_FROM_EXPERIMENT manually.')
        return None
    if not _AUTO_RESUME_FROM_STAGE:
        return None
    auto_prev = _auto_experiment_name(
        _AUTO_RESUME_FROM_STAGE, KEEP_TOP_N_CLASSES, DROP_CLASSES, NUM_EPOCHS
    )
    candidate = os.path.join(EXPERIMENTS_DIR, auto_prev, 'best.pt')
    if os.path.exists(candidate):
        return candidate
    legacy = os.path.join(CKPT_DIR, f'{_AUTO_RESUME_FROM_STAGE}_best.pt')
    return legacy if os.path.exists(legacy) else None

# --- Basic setup ---
os.makedirs(MASKS_CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

assert os.path.isdir(YOUR_IMAGES_DRIVE), f'Missing: {YOUR_IMAGES_DRIVE}'

# --- Image sync Drive → SSD (skippable) ---
if SKIP_IMAGE_SYNC:
    YOUR_IMAGES_DIR = YOUR_IMAGES_DRIVE
    print('⏭️  SKIP_IMAGE_SYNC=True — using images directly from Drive (slow).')
else:
    _sync_marker = os.path.join(LOCAL_IMAGES_DIR, '.sync_done')
    _need_sync = not os.path.exists(_sync_marker)
    if not _need_sync:
        n_drive = sum(1 for f in os.listdir(YOUR_IMAGES_DRIVE)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        n_local = sum(1 for f in os.listdir(LOCAL_IMAGES_DIR)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        if n_drive != n_local:
            print(f'Image count drift ({n_drive} vs {n_local}) → re-syncing')
            _need_sync = True
    if _need_sync:
        os.makedirs(LOCAL_IMAGES_DIR, exist_ok=True)
        print(f'Syncing images Drive → {LOCAL_IMAGES_DIR} (one-time, ~3-5 min)...')
        t0 = time.time()
        # Find all image files recursively and copy to flat local dir
        # Also strips double-extension artifact (.JPG.JPG → .JPG)
        import shutil as _shutil, re as _re
        _img_exts = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
        _n_copied = 0
        for _r, _dirs, _fnames in os.walk(YOUR_IMAGES_DRIVE):
            for _fn in _fnames:
                _ext = os.path.splitext(_fn)[1]
                if _ext not in _img_exts: continue
                _src = os.path.join(_r, _fn)
                # Strip double extension: photo.JPG.JPG → photo.JPG
                _stem, _e = os.path.splitext(_fn)
                _stem2, _e2 = os.path.splitext(_stem)
                if _e2.lower() in {'.jpg','.jpeg','.png'}:
                    _dest_name = _stem2 + _e2  # was .JPG.JPG, keep first ext
                else:
                    _dest_name = _fn
                _dst = os.path.join(LOCAL_IMAGES_DIR, _dest_name)
                if not os.path.exists(_dst):
                    _shutil.copy2(_src, _dst)
                    _n_copied += 1
        print(f'  Copied {_n_copied} new files')
        n_local = sum(1 for f in os.listdir(LOCAL_IMAGES_DIR)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        with open(_sync_marker, 'w') as f:
            f.write(f'synced {n_local} files at {time.time()}')
        print(f'✅ Synced {n_local} images in {time.time() - t0:.0f}s')
    else:
        n_local = sum(1 for f in os.listdir(LOCAL_IMAGES_DIR)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        print(f'✅ Images synced at {LOCAL_IMAGES_DIR} ({n_local} files)')
# --- Auto-resize oversized images on SSD (keeps Drive originals untouched) ---
# Images larger than MAX_LONG_EDGE are slow to load every epoch. This resizes
# them once on the local SSD copy so training is fast.
# render_coco_to_png checks actual vs COCO dimensions and rescales polygons,
# so COCO annotations stay aligned even if the image was resized here.
# Skipped when SKIP_IMAGE_SYNC=True (we're reading directly from Drive).
MAX_LONG_EDGE = 1500
if not SKIP_IMAGE_SYNC:
    _oversized = [
        f for f in os.listdir(LOCAL_IMAGES_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    _n_resized = 0
    for _fn in _oversized:
        _p = os.path.join(LOCAL_IMAGES_DIR, _fn)
        try:
            _img = cv2.imread(_p)
            if _img is None: continue
            _h, _w = _img.shape[:2]
            if max(_h, _w) > MAX_LONG_EDGE:
                _scale = MAX_LONG_EDGE / max(_h, _w)
                _img = cv2.resize(_img, (int(_w*_scale), int(_h*_scale)), interpolation=cv2.INTER_AREA)
                cv2.imwrite(_p, _img, [cv2.IMWRITE_JPEG_QUALITY, 90])
                _n_resized += 1
        except Exception as _e:
            print(f'  ⚠️ resize failed for {_fn}: {_e}')
    if _n_resized:
        print(f'✅ Auto-resized {_n_resized} oversized images to max {MAX_LONG_EDGE}px (SSD only)')
    else:
        print(f'✅ All images already ≤ {MAX_LONG_EDGE}px — no resize needed')
    # Always point training at the local SSD copy (regardless of whether resize ran)
    YOUR_IMAGES_DIR = LOCAL_IMAGES_DIR

# --- Summary print ---
_resume_path = _resolve_resume_path()
print('═' * 70)
print(f'🧪 EXPERIMENT: {EXPERIMENT_NAME}')
print(f'   Folder    : {EXPERIMENT_DIR}')
print('─' * 70)
print(f'Stage            : {STAGE}')
print(f'Resume from      : {_resume_path or "pretrained Coralscapes"}')
print(f'Top-N classes    : {KEEP_TOP_N_CLASSES if KEEP_TOP_N_CLASSES else "ALL"}')
print(f'Drop classes     : {DROP_CLASSES if DROP_CLASSES else "(none)"}')
print(f'Model            : {MODEL_NAME}   input={INPUT_SIZE}px   batch={BATCH_SIZE}')
print(f'Epochs           : {NUM_EPOCHS}   LR={LR}   weight_decay={WEIGHT_DECAY}')
print(f'Coralscapes head : pretrained={USE_PRETRAINED_CORAL}   joint-train={USE_CORALSCAPES_DATASET}')
print(f'Images           : {YOUR_IMAGES_DIR}')
print('═' * 70)

In [ ]:
# ===========================================================================
# CELL 3: (Optional) Download Coralscapes DATASET from HuggingFace
# ===========================================================================
# Downloads the full 2000-image Coralscapes dataset (images + masks) and dumps
# them to a local flat folder for fast reading during training.
#
# This is ONLY needed for 2-head regularization training (Head A uses these
# batches to keep the encoder anchored). It is NOT needed for the pretrained
# MODEL weights — those are downloaded separately in Cell 7 (~100 MB).
#
# If USE_CORALSCAPES_DATASET = False → this cell is a no-op.
# Cached across re-runs via a marker file (.export_done).
# ===========================================================================

def export_coralscapes_to_png(repo, cache_dir, images_dir, masks_dir, classes_txt):
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(masks_dir, exist_ok=True)

    done_marker = os.path.join(cache_dir, '.export_done')
    if os.path.exists(done_marker):
        print(f'✅ Coralscapes dataset cache already exists at {cache_dir} — skipping')
        return

    print(f'Downloading {repo} from HuggingFace (first time only) ...')
    from datasets import load_dataset
    ds = load_dataset(repo)
    print(f'Splits: {list(ds.keys())}')

    first_split = list(ds.keys())[0]
    sample = ds[first_split][0]
    print(f'Fields: {list(sample.keys())}')

    mask_field = None
    for c in ['label', 'mask', 'segmentation', 'semantic_segmentation', 'annotation']:
        if c in sample: mask_field = c; break
    assert mask_field, f'Could not find mask field in {list(sample.keys())}'
    img_field = 'image' if 'image' in sample else list(sample.keys())[0]
    print(f'Image field: "{img_field}"  Mask field: "{mask_field}"')

    class_names = None
    feat = ds[first_split].features.get(mask_field)
    if feat is not None and hasattr(feat, 'names') and feat.names:
        class_names = list(feat.names)

    from PIL import Image
    n = 0; max_idx = 0
    for split_name, split_ds in ds.items():
        print(f'  Split "{split_name}" ({len(split_ds)} samples)...')
        for i, ex in enumerate(split_ds):
            img = ex[img_field]
            mask = ex[mask_field]
            if not isinstance(img, Image.Image): img = Image.fromarray(np.array(img))
            if not isinstance(mask, Image.Image): mask = Image.fromarray(np.array(mask))
            stem = f'{split_name}_{i:06d}'
            img.convert('RGB').save(os.path.join(images_dir, stem + '.jpg'), quality=92)
            m_arr = np.array(mask).astype(np.uint8)
            max_idx = max(max_idx, int(m_arr.max()) if m_arr.size else 0)
            Image.fromarray(m_arr).save(os.path.join(masks_dir, stem + '.png'))
            n += 1
            if (i + 1) % 500 == 0: print(f'    {i+1}/{len(split_ds)}')
    print(f'Exported {n} image/mask pairs. Max class index: {max_idx}')

    if class_names is None:
        class_names = [f'class_{i}' for i in range(max_idx + 1)]
    with open(classes_txt, 'w') as f:
        f.write('\n'.join(class_names))
    print(f'Wrote {len(class_names)} class names → {classes_txt}')

    with open(done_marker, 'w') as f:
        f.write('done')


if USE_CORALSCAPES_DATASET:
    try:
        export_coralscapes_to_png(
            CORALSCAPES_HF_REPO,
            CORALSCAPES_CACHE_DIR,
            CORALSCAPES_IMAGES,
            CORALSCAPES_MASKS,
            CORALSCAPES_CLASSES_TXT,
        )
    except Exception as e:
        print(f'⚠️ Coralscapes download failed: {e}')
        print('   If auth error: run `from huggingface_hub import notebook_login; notebook_login()` first.')
        print('   Continuing in single-head mode (pretrained weights still used in Cell 7).')
        USE_CORALSCAPES_DATASET = False
        HAS_CORALSCAPES = False
else:
    print('Coralscapes DATASET skipped (USE_CORALSCAPES_DATASET=False).')
    print('→ Pretrained Coralscapes MODEL will still be loaded in Cell 7 (recommended).')

In [ ]:
# ===========================================================================
# CELL 4: Class taxonomies — one per head, never remapped
# ===========================================================================
# Head A (Coralscapes): 39 classes from the HF pretrained model (fixed)
# Head B (your data):   derived from your CSV's "Label code" column
# Keeping them separate means no information loss from forced remapping.
#
# Unknown / sentinel classes: any label listed in DROP_CLASSES below is
# removed from YOUR_CLASSES. In Cell 5, annotations with a dropped label are
# skipped (the pixel stays IGNORE_INDEX), so the model never learns to
# predict "I don't know" — it just ignores those points during training.
#
# Optional top-N filter: set KEEP_TOP_N_CLASSES in Cell 2 to keep only the N
# most common classes. Rare classes with <20 samples usually just add noise
# and drag down pixel mIoU. Dropped rare labels are also ignored, not
# relabeled. Leave as None to keep all classes.

import colorsys

# --- Classes to exclude from training (configurable in Cell 2) ---
# Case-insensitive match against CSV 'Label code'.
DROP_CLASSES = globals().get('DROP_CLASSES', [
    'Unknown', 'Unkn', 'Unk',
    'Off', '?', 'NA', 'nan', 'None', '',
])
DROP_CLASSES_LOWER = {c.strip().lower() for c in DROP_CLASSES}

# --- Optional top-N cap (configurable in Cell 2) ---
KEEP_TOP_N_CLASSES = globals().get('KEEP_TOP_N_CLASSES', None)  # None = keep all

# --- Head B: your classes (from CSV) ---
df_full = pd.read_csv(YOUR_CSV_PATH, low_memory=False)
label_col = 'Label code' if 'Label code' in df_full.columns else 'Label'

# Filter out drop-list labels first
_series = df_full[label_col].astype(str)
_keep_mask = ~_series.str.strip().str.lower().isin(DROP_CLASSES_LOWER)
_dropped_found = sorted(set(_series[~_keep_mask].unique()))
_n_dropped_anns = int((~_keep_mask).sum())

# Count per class
_class_counts = _series[_keep_mask].value_counts()



# ── Apply 2025-07 label merge ────────────────────────────────────────────
if USE_LABEL_MERGE:
    _remapped = _series.map(apply_label_merge)
    _keep_merge = _remapped.notna()
    _extra_dropped = int((~_keep_merge & _keep_mask).sum())
    _series = _remapped[_keep_merge & _keep_mask]
    _keep_mask = _keep_mask & _keep_merge
    _class_counts = _series.value_counts()
    if _extra_dropped:
        print(f'  Label merge: {_extra_dropped} annotations excluded (Unknown/MA/etc.)')
    print(f'  Merged classes: {len(_class_counts)} '
          f'(was {len(df_full[label_col].astype(str).value_counts())} original)')

# Apply top-N if requested
if KEEP_TOP_N_CLASSES is not None and KEEP_TOP_N_CLASSES < len(_class_counts):
    _top = set(_class_counts.head(KEEP_TOP_N_CLASSES).index)
    _rare_dropped = sorted(set(_class_counts.index) - _top)
    _n_rare_dropped_anns = int(_class_counts.loc[_rare_dropped].sum())
    YOUR_CLASSES = sorted(_top)
    print(f'  🔝 KEEP_TOP_N_CLASSES={KEEP_TOP_N_CLASSES} → dropped {len(_rare_dropped)} rare '
          f'classes ({_n_rare_dropped_anns} annotations become IGNORE).')
    print(f'     Rare dropped (sample): {_rare_dropped[:10]}')
else:
    YOUR_CLASSES = sorted(_class_counts.index.tolist())

YOUR_CLASS_TO_IDX = {c: i for i, c in enumerate(YOUR_CLASSES)}
YOUR_IDX_TO_CLASS = {i: c for c, i in YOUR_CLASS_TO_IDX.items()}
N_CLASSES_B = len(YOUR_CLASSES)
print(f'Head B (your data):  {N_CLASSES_B} classes — {YOUR_CLASSES[:6]} ...')
if _dropped_found:
    print(f'  🗑️  Sentinel labels dropped: {_dropped_found}  '
          f'→ {_n_dropped_anns} annotations IGNORED')

# --- Head A: Coralscapes classes ---
if HAS_CORALSCAPES and os.path.exists(CORALSCAPES_CLASSES_TXT):
    with open(CORALSCAPES_CLASSES_TXT) as f:
        CORALSCAPES_CLASSES = [l.strip() for l in f if l.strip()]
else:
    CORALSCAPES_CLASSES = [f'coralscape_{i}' for i in range(39)]
CORALSCAPES_CLASS_TO_IDX = {c: i for i, c in enumerate(CORALSCAPES_CLASSES)}
N_CLASSES_A = len(CORALSCAPES_CLASSES)
print(f'Head A (Coralscapes): {N_CLASSES_A} classes')

# --- Distinct color palettes (golden-ratio HSV — no two greens alike) ---
_GOLDEN_HSV = 0.61803398875

def make_palette(n, seed=0):
    """Golden-ratio HSV palette — maximally distinct colors for any N.

    Random RGB palettes cluster in green/brown (terrible for coral imagery
    where backgrounds are already those hues). Hue-spread with alternating
    saturation/value gives clear separation even at 80+ classes.
    """
    h0 = (seed * 0.137 + 0.12) % 1.0  # start off-green
    out = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        h = (h0 + i * _GOLDEN_HSV) % 1.0
        s = 0.70 + 0.25 * ((i * 7) % 3) / 2
        v = 0.72 + 0.25 * ((i * 11) % 2)
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        out[i] = [int(r * 255), int(g * 255), int(b * 255)]
    return out

PALETTE_A = make_palette(N_CLASSES_A, seed=1)
PALETTE_B = make_palette(N_CLASSES_B, seed=2)

In [ ]:
# ===========================================================================
# CELL 5: Render training target masks as PNGs (with cache)
# ===========================================================================
# Converts your label source (CSV points, SAM COCO, or pseudo-labels) into
# per-pixel class-index PNGs that the Dataset (Cell 6) will load.
#
#   points stage  → small disk per annotation (rest = 255 ignore)
#   sam_masks     → polygons rasterized, uncovered pixels = 255
#   pseudo        → combines previous labels + confident model predictions
#
# PNG value convention: uint8, pixel = class index (0..K-1), 255 = ignore.
#
# CACHING: after a successful render, we drop a .render_done fingerprint file.
# On re-run, if the fingerprint matches (source file mtime + params), we skip
# the whole render step. Set FORCE_RERENDER=True to force a rebuild.
#
# Masks live on local VM SSD — if the runtime restarts, cache is lost and
# re-rendered automatically. Set to Drive to make it truly persistent.

FORCE_RERENDER = globals().get("FORCE_RERENDER", False)
POINT_RADIUS   = 12  # ~5px disk after resize to 512; radius 3 was sub-pixel on 1300px images

from PIL import Image

def _get_image_size(path):
    """Read just image dimensions (no pixel decode). ~100× faster than cv2.imread."""
    try:
        with Image.open(path) as im:
            return im.size  # (w, h)
    except Exception:
        return None


def _fingerprint(source_path, extra_params):
    """Cheap cache key: source file mtime + size + param dict."""
    import hashlib
    st = os.stat(source_path)
    payload = f'{st.st_mtime}|{st.st_size}|{json.dumps(extra_params, sort_keys=True)}'
    return hashlib.md5(payload.encode()).hexdigest()


def _is_cache_valid(out_dir, expected_fp):
    """True if cache dir exists, has .render_done with matching fingerprint, and >=1 PNG."""
    marker = os.path.join(out_dir, '.render_done')
    if not os.path.isfile(marker):
        return False
    with open(marker) as f:
        stored = f.read().strip()
    if stored != expected_fp:
        return False
    # quick file count
    n_pngs = sum(1 for f in os.listdir(out_dir) if f.endswith('.png'))
    return n_pngs > 0


def _write_cache_marker(out_dir, fp, count):
    with open(os.path.join(out_dir, '.render_done'), 'w') as f:
        f.write(fp)
    with open(os.path.join(out_dir, '.render_info.json'), 'w') as f:
        json.dump({'count': count, 'fingerprint': fp, 'ts': time.time()}, f)


import os as _os

# ── Smart image resolver ──────────────────────────────────────────────────────
# Handles case differences, double extensions, Roboflow hash suffixes, and
# -JPG/-jpg artifacts. Built once per images_dir, reused across all renders.

def _normalise_stem(name):
    """Strip all Roboflow/encoding artifacts and return a lowercase bare stem."""
    import re as _re
    # Strip .rf.HASH (anywhere in name, before extension)
    name = _re.sub(r'\.rf\.[A-Za-z0-9]+', '', name)
    # Strip trailing _JPG_JPG, -JPG_JPG, _jpg_jpg etc. (Roboflow double-encode)
    name = _re.sub(r'[-_](jpe?g|png)[-_](jpe?g|png)$', '', name, flags=_re.IGNORECASE)
    # Strip extension(s)
    stem = name
    for _ in range(3):  # handle up to triple extension
        base, ext = _os.path.splitext(stem)
        if ext.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}:
            stem = base
        else:
            break
    # Strip trailing -JPG/-jpg/_JPG/_jpg artifact on stem
    stem = _re.sub(r'[-_](jpe?g|png)$', '', stem, flags=_re.IGNORECASE)
    return stem.lower()


def build_image_index(images_dir):
    """
    Build a dict: normalised_stem -> list of actual filenames (not full paths).
    Call once per directory; pass the result to resolve_image().
    """
    import os as _os
    _img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff',
                 '.JPG', '.JPEG', '.PNG', '.BMP', '.TIFF'}
    index = {}
    for fn in _os.listdir(images_dir):
        if _os.path.splitext(fn)[1] not in _img_exts:
            continue
        key = _normalise_stem(fn)
        index.setdefault(key, []).append(fn)
    return index


def resolve_image(fname, images_dir, index):
    """
    Return full path to the best matching file for fname, or None if not found.
    index must be built with build_image_index(images_dir).

    Disambiguation priority (when multiple files share the same normalised stem):
      1. Exact filename match (case-insensitive)
      2. Shortest filename (least decoration)
      3. Alphabetically first
    """
    import os as _os
    key = _normalise_stem(fname)
    candidates = index.get(key, [])
    if not candidates:
        return None
    if len(candidates) == 1:
        return _os.path.join(images_dir, candidates[0])
    # Prefer exact case-insensitive match
    fname_lower = fname.lower()
    for c in candidates:
        if c.lower() == fname_lower:
            return _os.path.join(images_dir, c)
    # Prefer shortest name (least Roboflow decoration)
    best = sorted(candidates, key=lambda x: (len(x), x))[0]
    return _os.path.join(images_dir, best)


def render_points_to_png(csv_path, images_dir, out_dir, class_to_idx, ignore=255,
                         label_col='Label code', point_radius=3):
    """Sparse rendering: small disk per point; rest = ignore."""
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path, low_memory=False)
    df = df.rename(columns={label_col: 'Label'})
    df = df[['Name', 'Row', 'Column', 'Label']].dropna()
    df['Row'] = df['Row'].astype(int); df['Column'] = df['Column'].astype(int)

    n_done = 0
    for name, g in df.groupby('Name'):
        if '_pts_img_index' not in dir():
            _pts_img_index = build_image_index(images_dir)
        img_path = resolve_image(name, images_dir, _pts_img_index)
        if not img_path: continue
        size = _get_image_size(img_path)
        if size is None: continue
        w, h = size
        mask = np.full((h, w), ignore, dtype=np.uint8)
        for _, r in g.iterrows():
            lbl = str(r['Label'])
            if lbl not in class_to_idx: continue
            ci = class_to_idx[lbl]
            y = min(max(int(r['Row']), 0), h - 1)
            x = min(max(int(r['Column']), 0), w - 1)
            if point_radius > 0:
                cv2.circle(mask, (x, y), point_radius, ci, -1)
            else:
                mask[y, x] = ci
        out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(img_path))[0] + '.png')
        cv2.imwrite(out_path, mask)
        n_done += 1
        if n_done % 500 == 0:
            print(f'  ... {n_done} masks written')
    print(f'  Rendered {n_done} point-level masks → {out_dir}')
    return n_done



# ── Roboflow COCO preprocessor (runs automatically for gold_masks + sam_masks)
# Handles raw Roboflow exports so you never need a separate fix script.
GENUS_TO_CORALNET = {
    # Hard corals — full scientific name → CoralNet short code
    'Acanthastrea': 'Aca',      'Acropora': 'Acr',          'Alveopora': 'Alv',
    'Anomastraea': 'Ano',       'Astrea': 'Astrea',         'Astreopora': 'Astreo',
    'Blastomussa': 'Bla',       'Caulastrea': 'Caula',      'Coscinaraea': 'Cos',
    'Cyphastrea': 'Cyph',       'Diploastrea': 'Diplo',     'Dipsastraea': 'Dipsa',
    'Duncanopsammia': 'Dun',    'Echinophyllia': 'Echphy',  'Echinopora': 'Echpo',
    'Favites': 'Favit',         'Fungiidae': 'Fungii',      'Galaxea': 'Gal',
    'Gardineroseris': 'Gar',    'Goniastrea': 'Gonia',      'Goniopora': 'Gonio',
    'Hard coral': 'HC',         'Hydnophora': 'Hydno',      'Isopora': 'Iso',
    'Leptastrea': 'Lepta',      'Leptoria': 'Leptor',       'Leptoseris': 'Leptos',
    'Lobophyllia': 'Lobophyl',  'Merulina': 'Mer',          'Micromussa': 'Micro',
    'Montipora': 'Monti',       'Mycedium': 'Myc',          'Oulophyllia': 'Oul',
    'Oxypora': 'Oxy',           'Pachyseris': 'Pachy',      'Paramontastea': 'Para',
    'Paramontastraea': 'Para',  'Pavona': 'Pav',            'Pectinia': 'Pec',
    'Physogyra': 'Phy',         'Platygyra': 'Platy',       'Plerogyra': 'Plero',
    'Plesiastrea': 'Plesia',    'Pocillopora': 'Poc',       'Podabacia': 'Pod',
    'Porites (branching)': 'PorB', 'Porites branching': 'PorB',
    'Porites (massive)': 'PorM',   'Porites massive': 'PorM',
    'Psammocora': 'Psa',        'Seriatopora': 'Ser',       'Stylophora': 'Styl',
    'Symphyllia': 'Sym',        'Turbinaria (coral)': 'Turb-HC', 'Turbinaria': 'Turb-HC',
    # Other invertebrates
    'Aglaophenia spp.': 'Agl',  'Anemone': 'Anemone',       'Bivalve': 'Bivalve',
    'Bryozoan': 'Bryo',         'Corallimorpharia': 'Cormor', 'Didemnidae': 'Didae',
    'Hydroid': 'Hydro',         'Lobophytum': 'Lobphyt',    'Millepora': 'Mil',
    'Sarcophyton': 'Sarcopydae','Soft Coral': 'SC',         'Sponge': 'SP',
    'Tubipora musica': 'Tubmus','Tunicate': 'Tun',          'Echinoderms: sea urchin': 'Urchins ex',
    'XENIIDAE': 'Xe',           'Zoanthid': 'Zoan',
    # Substrates / other
    'Biofilm': 'Biofilm ex',    'Rhytisma': 'Rhy',          'Sand': 'S',
    'Dead coral': 'Dead (ex)',  'Hard Substrate': 'HS_AR',  'Rock': 'Rock',
    'Cyanobacteria': 'Cya',     'Framer': 'Frame',          'Rubble': 'R',
    'Unknown': 'Unknown',
    # Algae
    'Ochrophyta': 'BA',         'Caulerpa': 'Cau',          'CCA (crustose coralline algae)': 'CCA',
    'Dictyota': 'Dic',          'Algae (filamentous)': 'Fila (ex)', 'Chlorophyta': 'GA',
    'Halimeda': 'Hal',          'Lobophora variegata': 'Lobvar',   'Macroalgae': 'MA',
    'Padina': 'Pad',            'Rhodophyta': 'RA',         'Sargassum': 'Sar',
    'Turf algae': 'TA',         'Turbinaria (algae)': 'Turb-BA',   'Valonia spp.': 'Val',
    'Seagrass': 'SG',
    # Roboflow non-class — drop
    'REEFo': None,
}

def _clean_roboflow_filename(fname):
    """Strip Roboflow artifacts from filenames: .rf.HASH, -JPG suffix, double extension."""
    import re as _re, os as _os
    # Strip .rf.HASH suffix (before extension)
    fname = _re.sub(r'\.rf\.[A-Za-z0-9]+', '', fname)
    stem, ext = _os.path.splitext(fname)
    # Strip double extension: photo.JPG.JPG → photo.JPG
    stem2, ext2 = _os.path.splitext(stem)
    if ext2.lower() in {'.jpg', '.jpeg', '.png'}:
        stem, ext = stem2, ext2
    # Strip -JPG/-jpg/-JPEG/-PNG artifact appended to stem
    stem = _re.sub(r'[-_](jpe?g|png)$', '', stem, flags=_re.IGNORECASE)
    return stem + ext

def preprocess_roboflow_coco(coco_path):
    """
    Load a COCO JSON and fix Roboflow artifacts in-memory. Returns cleaned coco dict.
    Safe to call on already-clean files — no-ops if nothing needs fixing.
    Fixes:
      - File names: strips -JPG/-jpg artifact and .rf.HASH suffixes
      - Category names: remaps full scientific names to CoralNet short codes
      - Drops unused categories (e.g. REEFo)
    """
    import os as _os
    with open(coco_path, encoding='utf-8') as _f:
        coco = json.load(_f)

    # Fix file names
    n_renamed = 0
    for img in coco['images']:
        # Prefer extra.name if present
        clean = img.get('extra', {}).get('name', '')
        if not clean:
            clean = _clean_roboflow_filename(img['file_name'])
        else:
            clean = _clean_roboflow_filename(clean)
        if clean != img['file_name']:
            img['file_name'] = clean
            n_renamed += 1

    # Remap category names
    old_cats = {c['id']: c['name'] for c in coco.get('categories', [])}
    needs_remap = any(nm in GENUS_TO_CORALNET for nm in old_cats.values())
    if globals().get('USE_LABEL_MERGE', False):
        needs_remap = needs_remap or any(
            apply_label_merge(nm) != nm for nm in old_cats.values() if nm)

    if needs_remap:
        seen, new_cats, old_to_new = {}, [], {}
        for c in coco['categories']:
            new_name = GENUS_TO_CORALNET.get(c['name'], c['name'])
            # Also apply label merge so e.g. 'Cyph' → 'HC' matches the trained merged classes
            if new_name is not None and globals().get('USE_LABEL_MERGE', False):
                _merged = apply_label_merge(new_name)
                if _merged is not None:
                    new_name = _merged
            if new_name is None:
                old_to_new[c['id']] = None
                continue
            if new_name not in seen:
                new_id = len(new_cats) + 1
                seen[new_name] = new_id
                new_cats.append({'id': new_id, 'name': new_name, 'supercategory': 'coral'})
            old_to_new[c['id']] = seen[new_name]
        coco['categories'] = new_cats
        kept, dropped = 0, 0
        new_anns = []
        for a in coco['annotations']:
            new_id = old_to_new.get(a['category_id'])
            if new_id is None:
                dropped += 1
            else:
                a['category_id'] = new_id
                new_anns.append(a)
                kept += 1
        coco['annotations'] = new_anns
        print(f'  COCO preprocessed: {n_renamed} filenames cleaned, '
              f'{len(old_cats)}→{len(new_cats)} categories, {dropped} annotations dropped')
    elif n_renamed:
        print(f'  COCO filenames cleaned: {n_renamed} files renamed')

    return coco

def render_coco_to_png(coco_path, images_dir, out_dir, class_to_idx, ignore=255):
    """Dense rendering from SAM COCO JSON. Each polygon filled with its class.
    Uncovered pixels = ignore. Warns if COCO categories don't match class_to_idx."""
    os.makedirs(out_dir, exist_ok=True)
    if isinstance(coco_path, dict):
        coco = coco_path
    else:
        coco = preprocess_roboflow_coco(coco_path)

    cat_id_to_name = {c['id']: c['name'] for c in coco.get('categories', [])}
    cat_id_to_idx  = {cid: class_to_idx.get(nm, None) for cid, nm in cat_id_to_name.items()}

    unknown = [nm for cid, nm in cat_id_to_name.items() if cat_id_to_idx[cid] is None]
    known   = [nm for cid, nm in cat_id_to_name.items() if cat_id_to_idx[cid] is not None]
    print(f'  COCO categories: {len(cat_id_to_name)} total → {len(known)} matched, {len(unknown)} dropped')
    if unknown:
        print(f'  ⚠️ Dropped COCO classes (not in your CSV — check whitespace/case):')
        for nm in unknown[:10]:
            print(f'     "{nm}"')
        if len(unknown) > 10:
            print(f'     ... and {len(unknown) - 10} more')

    images_by_id = {im['id']: im for im in coco['images']}
    anns_by_img = defaultdict(list)
    for a in coco['annotations']:
        anns_by_img[a['image_id']].append(a)
    _img_index = build_image_index(images_dir)

    n_done = 0
    n_anns_kept = 0
    n_anns_dropped = 0
    _rescale_warnings = {}
    for img_id, img_meta in images_by_id.items():
        fname = img_meta['file_name']
        img_path = resolve_image(fname, images_dir, _img_index)
        if not img_path: continue
        fname = os.path.basename(img_path)
        coco_h, coco_w = img_meta['height'], img_meta['width']
        # Check actual file dimensions — COCO may have been exported from differently-sized images
        with Image.open(img_path) as _im:
            actual_w, actual_h = _im.size
        sx = actual_w / coco_w if coco_w else 1.0
        sy = actual_h / coco_h if coco_h else 1.0
        if abs(sx - 1.0) > 0.01 or abs(sy - 1.0) > 0.01:
            key = f'{coco_w}x{coco_h}->{actual_w}x{actual_h}'
            _rescale_warnings[key] = _rescale_warnings.get(key, 0) + 1
        h, w = actual_h, actual_w
        mask = np.full((h, w), ignore, dtype=np.uint8)
        anns = sorted(anns_by_img[img_id], key=lambda a: -a.get('area', 0))
        for a in anns:
            ci = cat_id_to_idx.get(a['category_id'])
            if ci is None:
                n_anns_dropped += 1
                continue
            _seg = a.get('segmentation', [])
            # Handle RLE format (dict with 'counts'/'size') vs polygon format (list of lists)
            if isinstance(_seg, dict):
                try:
                    from pycocotools import mask as _maskutil
                    _rle = _seg
                    if isinstance(_rle.get('counts'), list):
                        # Uncompressed RLE — convert to compressed first
                        _rle = _maskutil.frPyObjects([_rle], _rle['size'][0], _rle['size'][1])[0]
                    _bin = _maskutil.decode(_rle).astype(np.uint8)
                    if _bin.shape != mask.shape:
                        _bin = cv2.resize(_bin, (mask.shape[1], mask.shape[0]),
                                          interpolation=cv2.INTER_NEAREST)
                    mask[_bin > 0] = ci
                except Exception as _e:
                    print(f'  ⚠️ RLE decode failed for ann {a.get("id")}: {_e}')
                    continue
            else:
                for poly in _seg:
                    if len(poly) < 6: continue
                    pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                    pts[:, 0] *= sx
                    pts[:, 1] *= sy
                    cv2.fillPoly(mask, [pts.astype(np.int32)], ci)
            n_anns_kept += 1
        out_path = os.path.join(out_dir, os.path.splitext(fname)[0] + '.png')
        cv2.imwrite(out_path, mask)
        n_done += 1
    if _rescale_warnings:
        print(f'  ⚠️  COCO dimension mismatch — annotations rescaled automatically:')
        for desc, cnt in _rescale_warnings.items():
            print(f'     {desc} × {cnt} images')
    _n_total_imgs = len(images_by_id)
    _n_skipped = _n_total_imgs - n_done
    print(f'  Rendered {n_done} COCO masks → {out_dir}')
    if _n_skipped > 0:
        print(f'  ⚠️  {_n_skipped}/{_n_total_imgs} COCO images had NO matching file in {out_dir}.')
        print(f'     These annotations will be SKIPPED in training.')
        print(f'     Upload missing images to Drive → {images_dir}')
    print(f'  Annotations: {n_anns_kept} kept, {n_anns_dropped} dropped')
    return n_done


# --- Render for the current stage (with cache) ---
stage_mask_dir = os.path.join(MASKS_CACHE_DIR, STAGE)
os.makedirs(stage_mask_dir, exist_ok=True)

if STAGE == 'points':
    params = {'point_radius': POINT_RADIUS, 'label_col': label_col,
              'n_classes': len(YOUR_CLASS_TO_IDX), 'version': 1}
    fp = _fingerprint(YOUR_CSV_PATH, params)

    if not FORCE_RERENDER and _is_cache_valid(stage_mask_dir, fp):
        n_cached = sum(1 for f in os.listdir(stage_mask_dir) if f.endswith('.png'))
        print(f'✅ Cache hit: {n_cached} existing point masks at {stage_mask_dir} (skipping render)')
        print(f'   Set FORCE_RERENDER=True above to rebuild.')
    else:
        reason = 'FORCE_RERENDER=True' if FORCE_RERENDER else 'cache miss or CSV changed'
        print(f'Rendering point-level masks ({reason}) ...')
        n = render_points_to_png(
            YOUR_CSV_PATH, YOUR_IMAGES_DIR, stage_mask_dir,
            YOUR_CLASS_TO_IDX, ignore=IGNORE_INDEX, label_col=label_col,
            point_radius=POINT_RADIUS,
        )
        _write_cache_marker(stage_mask_dir, fp, n)

elif STAGE == 'sam_masks':
    assert os.path.exists(YOUR_SAM_COCO_PATH), \
        f'SAM COCO not found: {YOUR_SAM_COCO_PATH}  — run batch_export_sam.ipynb first'

    # --- Sanity check: show when/how this sam_coco.json was made ---
    # Lets you verify you're training on the SAM export you just made, not an
    # older one with different settings. Prints timestamp + AMG params.
    import datetime as _dt
    _mtime = _dt.datetime.fromtimestamp(os.path.getmtime(YOUR_SAM_COCO_PATH))
    _size_mb = os.path.getsize(YOUR_SAM_COCO_PATH) / 1e6
    print(f'→ Using SAM COCO: {YOUR_SAM_COCO_PATH}')
    print(f'  File size:   {_size_mb:.1f} MB')
    print(f'  Modified:    {_mtime:%Y-%m-%d %H:%M:%S} (local time)')
    try:
        with open(YOUR_SAM_COCO_PATH) as _f:
            _info = json.load(_f).get('info', {})
        print(f'  Created:     {_info.get("date_created", "?")}')
        _ps = _info.get('pipeline_settings', {}) or {}
        print(f'  SAM model:   {_ps.get("sam_model", "?")}  '
              f'fp16={_ps.get("float16", "?")}')
        print(f'  AMG:         grid={_ps.get("amg_points_per_side", "?")}  '
              f'IoU>{_ps.get("amg_pred_iou_thresh", "?")}  '
              f'stab>{_ps.get("amg_stability_thresh", "?")}  '
              f'maxmask={_ps.get("max_mask_pct", "?")}%')
        print(f'  Images:      {_ps.get("total_images_processed", "?")}  '
              f'Annotations: {_ps.get("total_annotations", "?")}')
    except Exception as _e:
        print(f'  (could not read info header: {_e})')
    _hours_old = (_dt.datetime.now() - _mtime).total_seconds() / 3600
    if _hours_old > 24 * 7:
        print(f'  ⚠️  This COCO file is {_hours_old/24:.0f} days old — '
              f'is this really the latest SAM export?')

    params = {'n_classes': len(YOUR_CLASS_TO_IDX), 'version': 1}
    fp = _fingerprint(YOUR_SAM_COCO_PATH, params)

    if not FORCE_RERENDER and _is_cache_valid(stage_mask_dir, fp):
        n_cached = sum(1 for f in os.listdir(stage_mask_dir) if f.endswith('.png'))
        print(f'✅ Cache hit: {n_cached} existing COCO masks at {stage_mask_dir} (skipping render)')
        print(f'   Set FORCE_RERENDER=True above to rebuild.')
    else:
        reason = 'FORCE_RERENDER=True' if FORCE_RERENDER else 'cache miss or COCO changed'
        print(f'Rendering SAM COCO masks ({reason}) ...')
        n = render_coco_to_png(YOUR_SAM_COCO_PATH, YOUR_IMAGES_DIR, stage_mask_dir,
                                YOUR_CLASS_TO_IDX, ignore=IGNORE_INDEX)
        _write_cache_marker(stage_mask_dir, fp, n)


elif STAGE == 'gold_masks':
    # Stage 4: professionally annotated Roboflow COCO (same pipeline as sam_masks)
    _gold_path = globals().get('YOUR_GOLD_COCO_PATH', '')
    if not _gold_path or not os.path.exists(_gold_path):
        raise FileNotFoundError(
            "Stage 4 (gold_masks) requires YOUR_GOLD_COCO_PATH.\n"
            "Set it in Cell 2 and upload your Roboflow COCO export to Drive."
        )
    _gold_coco = preprocess_roboflow_coco(_gold_path)
    fp = _fingerprint(_gold_path, {'stage': 'gold_masks', 'classes': YOUR_CLASSES})
    if not FORCE_RERENDER and _is_cache_valid(stage_mask_dir, fp):
        n_cached = sum(1 for f in os.listdir(stage_mask_dir) if f.endswith('.png'))
        print(f'Cache hit: {n_cached} gold_masks PNGs (skipping render)')
    else:
        n = render_coco_to_png(_gold_coco, YOUR_IMAGES_DIR, stage_mask_dir,
                               YOUR_CLASS_TO_IDX)
        _write_cache_marker(stage_mask_dir, fp, n)
        print(f'Rendered {n} gold_masks PNGs -> {stage_mask_dir}')

elif STAGE == 'targeted':
    # Stage 5: targeted refinement — extra Roboflow COCO annotations of
    # classes that performed poorly in Cell 9's confusion matrix.
    # Same render pipeline as gold_masks, just a different file.
    _tgt_path = globals().get('YOUR_TARGETED_COCO_PATH', '')
    if not _tgt_path or not os.path.exists(_tgt_path):
        raise FileNotFoundError(
            "Stage 5 (targeted) requires YOUR_TARGETED_COCO_PATH.\n"
            "1. Decide which classes underperformed (Cell 9's confusion matrix).\n"
            "2. Annotate them in a NEW Roboflow project and export COCO JSON.\n"
            "3. Upload to Drive as the path printed above."
        )
    _tgt_coco = preprocess_roboflow_coco(_tgt_path)
    fp = _fingerprint(_tgt_path, {'stage': 'targeted', 'classes': YOUR_CLASSES})
    if not FORCE_RERENDER and _is_cache_valid(stage_mask_dir, fp):
        n_cached = sum(1 for f in os.listdir(stage_mask_dir) if f.endswith('.png'))
        print(f'Cache hit: {n_cached} targeted PNGs (skipping render)')
    else:
        n = render_coco_to_png(_tgt_coco, YOUR_IMAGES_DIR, stage_mask_dir,
                               YOUR_CLASS_TO_IDX)
        _write_cache_marker(stage_mask_dir, fp, n)
        print(f'Rendered {n} targeted PNGs -> {stage_mask_dir}')

elif STAGE == 'pseudo':
    stage_mask_dir = os.path.join(MASKS_CACHE_DIR, 'pseudo')
    assert os.path.isdir(stage_mask_dir), \
        f'Pseudo-label masks not found at {stage_mask_dir}. ' \
        f'Run Cell 13 in a previous stage to generate them.'

else:
    raise ValueError(f'Unknown STAGE: {STAGE}')

# Sanity check — peek at one mask
_samples = [f for f in os.listdir(stage_mask_dir) if f.endswith('.png')]
if _samples:
    m = cv2.imread(os.path.join(stage_mask_dir, _samples[0]), 0)
    uniq = np.unique(m)
    ignore_pct = (m == IGNORE_INDEX).sum() / m.size * 100
    print(f'\nSample mask "{_samples[0]}":  shape={m.shape},  unique values (first 15)={uniq[:15].tolist()}')
    print(f'  → {ignore_pct:.1f}% ignored pixels')
    if STAGE == 'points' and ignore_pct > 99.95:
        print('  ⚠️ > 99.95% ignored — point_radius may be too small for your image size.')

In [ ]:
# ===========================================================================
# CELL 5b: Visual sanity check — preview training data before running
# ===========================================================================
# Shows 3 sample images per active stage (2=patches/CSV, 4=gold COCO).
# Overlays annotations so you can confirm: right images, right labels,
# right scale. Stage 3 (SAM masks) shows rendered PNGs directly.
# Run this after Cell 5. Safe to skip — does not modify any data.

import random, textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

_PREVIEW_N = 3   # images per stage
import matplotlib; _CMAP = matplotlib.colormaps['tab20'].resampled(len(YOUR_CLASSES))

def _color(idx):
    r, g, b, _ = _CMAP(idx % 20)
    return (int(b*255), int(g*255), int(r*255))  # BGR for cv2

def _preview_stage2(n=_PREVIEW_N):
    """CSV point annotations overlaid on images."""
    import pandas as pd
    if not os.path.exists(YOUR_CSV_PATH):
        print("Stage 2 preview: CSV not found, skipping"); return
    df = pd.read_csv(YOUR_CSV_PATH, low_memory=False)
    df = df.rename(columns={label_col: 'Label'})
    if USE_LABEL_MERGE:
        df['Label'] = df['Label'].map(apply_label_merge)
        df = df[df['Label'].notna()]
    names = df['Name'].unique().tolist()
    random.shuffle(names)
    samples = names[:n]
    fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5))
    if len(samples) == 1: axes = [axes]
    fig.suptitle("Stage 2 — CSV point annotations", fontsize=13, fontweight='bold')
    for ax, name in zip(axes, samples):
        # find image file
        img_path = None
        if '_prev2_idx' not in dir(): _prev2_idx = build_image_index(YOUR_IMAGES_DIR)
        img_path = resolve_image(name, YOUR_IMAGES_DIR, _prev2_idx)
        if not img_path:
            ax.set_title(f"{name}\n(image not found)"); ax.axis('off'); continue
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        rows = df[df['Name'] == name]
        rows = rows[rows['Label'].map(lambda c: YOUR_CLASS_TO_IDX.get(c,-1) >= 0)]
        # Show full image with dots (small)
        _thumb = img.copy()
        for _, r in rows.iterrows():
            _ci = YOUR_CLASS_TO_IDX.get(r['Label'], 0)
            _col = [int(c*255) for c in _CMAP(_ci % 20)[:3]]
            cv2.circle(_thumb, (int(r['Column']), int(r['Row'])), 6, _col, -1)
            cv2.circle(_thumb, (int(r['Column']), int(r['Row'])), 6, (255,255,255), 1)
        ax.imshow(_thumb)
        ax.set_title(f"{name}\n{len(rows)} points, {rows['Label'].nunique()} classes", fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()
    print(f"Stage 2: showed {len(samples)} images")
    # Also show patch crops (64x64) so you can see what the model actually trains on
    import numpy as _np
    _all_rows = []
    for name in samples:
        if '_prev2_idx' not in dir(): _prev2_idx = build_image_index(YOUR_IMAGES_DIR)
        _ip = resolve_image(name, YOUR_IMAGES_DIR, _prev2_idx)
        if not _ip: continue
        _img = cv2.cvtColor(cv2.imread(_ip), cv2.COLOR_BGR2RGB)
        _rows = df[df['Name'] == name]
        _rows = _rows[_rows['Label'].map(lambda c: YOUR_CLASS_TO_IDX.get(c,-1) >= 0)]
        _rows = _rows.sample(min(len(_rows), 6), random_state=0) if len(_rows)>6 else _rows
        for _, _r in _rows.iterrows():
            _all_rows.append((_img, int(_r['Column']), int(_r['Row']), _r['Label']))
    _all_rows = _all_rows[:18]
    if _all_rows:
        _nc = min(len(_all_rows), 6)
        _nr = (_len := len(_all_rows), (_len + _nc - 1) // _nc)[1]
        _fig2, _axs = plt.subplots(_nr, _nc, figsize=(_nc*2.2, _nr*2.2))
        _fig2.suptitle('Stage 2 — 64×64 patch crops (training view)', fontsize=11, fontweight='bold')
        _axs = [_axs] if _len==1 else list(_np.array(_axs).flat)
        _R = 32
        for _pi, (_img, _cx, _cy, _lbl) in enumerate(_all_rows):
            _x0,_x1 = max(0,_cx-_R), min(_img.shape[1],_cx+_R)
            _y0,_y1 = max(0,_cy-_R), min(_img.shape[0],_cy+_R)
            _crop = _img[_y0:_y1, _x0:_x1]
            _ci = YOUR_CLASS_TO_IDX.get(_lbl, 0)
            _color = _CMAP(_ci % 20)[:3]
            _axs[_pi].imshow(_crop)
            _axs[_pi].set_title(_lbl, fontsize=8, color=_color, fontweight='bold')
            _axs[_pi].axis('off')
        for _pi in range(len(_all_rows), len(_axs)): _axs[_pi].axis('off')
        plt.tight_layout(); plt.show()

def _preview_stage4(n=_PREVIEW_N):
    """Gold COCO polygon outlines overlaid on images."""
    gold_path = globals().get('YOUR_GOLD_COCO_PATH', '')
    if not gold_path or not os.path.exists(gold_path):
        print("Stage 4 preview: gold COCO not found, skipping"); return
    coco = preprocess_roboflow_coco(gold_path)
    cat_id_to_name = {c['id']: c['name'] for c in coco.get('categories', [])}
    anns_by_img = {}
    for a in coco['annotations']:
        anns_by_img.setdefault(a['image_id'], []).append(a)
    imgs = list(coco['images'])
    random.shuffle(imgs)
    samples = imgs[:n]
    fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5))
    if len(samples) == 1: axes = [axes]
    fig.suptitle("Stage 4 — Gold COCO polygon annotations", fontsize=13, fontweight='bold')
    for ax, img_meta in zip(axes, samples):
        fname = img_meta['file_name']
        if '_prev4_idx' not in dir(): _prev4_idx = build_image_index(YOUR_IMAGES_DIR)
        img_path = resolve_image(fname, YOUR_IMAGES_DIR, _prev4_idx)
        if not img_path:
            ax.set_title(f"{fname}\n(image not found)"); ax.axis('off'); continue
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        # rescale if needed
        actual_h, actual_w = img.shape[:2]
        sx = actual_w / img_meta['width']  if img_meta['width']  else 1.0
        sy = actual_h / img_meta['height'] if img_meta['height'] else 1.0
        anns = anns_by_img.get(img_meta['id'], [])
        for a in anns:
            nm = cat_id_to_name.get(a['category_id'], '?')
            idx = YOUR_CLASS_TO_IDX.get(nm, 0)
            col = [int(c*255) for c in _CMAP(idx % 20)[:3]]
            _seg = a.get('segmentation', [])
            if isinstance(_seg, dict):
                # RLE format — decode and draw outline of mask
                try:
                    from pycocotools import mask as _maskutil
                    _rle = _seg
                    if isinstance(_rle.get('counts'), list):
                        _rle = _maskutil.frPyObjects([_rle], _rle['size'][0], _rle['size'][1])[0]
                    _bin = _maskutil.decode(_rle).astype(np.uint8)
                    if _bin.shape[:2] != img.shape[:2]:
                        _bin = cv2.resize(_bin, (img.shape[1], img.shape[0]),
                                          interpolation=cv2.INTER_NEAREST)
                    _contours, _ = cv2.findContours(_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(img, _contours, -1, col, 2)
                except Exception: continue
            else:
                for poly in _seg:
                    if len(poly) < 6: continue
                    pts = np.array(poly, dtype=np.float32).reshape(-1,2)
                    pts[:,0] *= sx; pts[:,1] *= sy
                    cv2.polylines(img, [pts.astype(np.int32)], True, col, 2)
        ax.imshow(img)
        ax.set_title(f"{fname}\n{len(anns)} annotations", fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()
    print(f"Stage 4: showed {len(samples)} images")

# Run previews for active stages only
_active = set(TRAINING_FLOW)
print(f"TRAINING_FLOW = \"{TRAINING_FLOW}\" — checking stages: {sorted(_active - {'1'})}")
if '2' in _active:
    _preview_stage2()
if '4' in _active:
    _preview_stage4()
if not (_active & {'2', '4'}):
    print("No stages with visual previews (only stage 3 active — rendered masks shown by Cell 5 sanity check)")


In [ ]:
# ===========================================================================
# CELL 6: Dataset + augmentations + dataloaders
# ===========================================================================
# CoralSegDataset pairs images with their rendered masks. Augmentations from
# albumentations apply geometric transforms to image AND mask together — so
# they can never drift out of alignment. The assert in __getitem__ is a
# hard guarantee that shapes match.
#
# A 90/10 train/val split is seeded (SEED=42) so it's the same every run.

# --- Augmentation pipelines ---
# Geometric transforms (resize, pad, crop, flip) apply to mask too.
# Pixel transforms (brightness, hue) apply only to image.
# Padding fills mask with IGNORE_INDEX so padded pixels are excluded from loss.
train_tf = A.Compose([
    A.LongestMaxSize(max_size=INPUT_SIZE * 2, interpolation=cv2.INTER_AREA),
    A.PadIfNeeded(min_height=INPUT_SIZE, min_width=INPUT_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, fill=0, mask_fill_value=IGNORE_INDEX),
    A.RandomCrop(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.HueSaturationValue(10, 15, 10, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.LongestMaxSize(max_size=INPUT_SIZE, interpolation=cv2.INTER_AREA),
    A.PadIfNeeded(min_height=INPUT_SIZE, min_width=INPUT_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, fill=0, mask_fill_value=IGNORE_INDEX),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class CoralSegDataset(Dataset):
    """Paired image + PNG mask. Asserts shape alignment every __getitem__."""
    def __init__(self, image_paths, mask_paths, dataset_id, transform):
        assert len(image_paths) == len(mask_paths)
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.dataset_id = dataset_id   # 0 = Head A (Coralscapes), 1 = Head B (your data)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_UNCHANGED)
        assert mask is not None, f'Failed to load mask: {self.mask_paths[idx]}'
        assert img.shape[:2] == mask.shape[:2], (
            f'Image/mask size mismatch for {self.image_paths[idx]}: '
            f'img={img.shape[:2]} vs mask={mask.shape[:2]}'
        )
        t = self.transform(image=img, mask=mask)
        return t['image'], t['mask'].long(), self.dataset_id


def build_pair_list(images_dir, masks_dir):
    """Return (image_paths, mask_paths) for files with matching stem in both folders."""
    exts = {'.jpg', '.jpeg', '.png'}
    img_map = {os.path.splitext(f)[0]: os.path.join(images_dir, f)
               for f in os.listdir(images_dir)
               if os.path.splitext(f)[1].lower() in exts}
    imgs, masks = [], []
    for mf in os.listdir(masks_dir):
        if not mf.endswith('.png'): continue
        stem = os.path.splitext(mf)[0]
        if stem in img_map:
            imgs.append(img_map[stem])
            masks.append(os.path.join(masks_dir, mf))
    return imgs, masks


def make_loader(image_paths, mask_paths, dataset_id, transform, shuffle, is_train=None):
    """DataLoader with quality-safe performance tuning.

    Val loaders use num_workers=0 so they can be re-iterated across cell runs
    without the "DataLoader worker exited unexpectedly" error that happens
    when Colab kills idle persistent workers between cells.
    """
    if is_train is None:
        is_train = shuffle
    ds = CoralSegDataset(image_paths, mask_paths, dataset_id, transform)
    if is_train:
        kwargs = dict(
            batch_size=BATCH_SIZE, shuffle=shuffle,
            num_workers=NUM_WORKERS, pin_memory=True,
            drop_last=shuffle,
        )
        if NUM_WORKERS > 0:
            kwargs['prefetch_factor'] = PREFETCH_FACTOR
            kwargs['persistent_workers'] = PERSISTENT_WORKERS
    else:
        # Val: no workers, no persistence → robust across interrupts / cell re-runs.
        kwargs = dict(
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=False, drop_last=False,
        )
    return DataLoader(ds, **kwargs)


def split_90_10(items_a, items_b, seed):
    """Seeded 90/10 split. Returns (train_a, train_b, val_a, val_b)."""
    rng = np.random.default_rng(seed)
    n = len(items_a)
    perm = rng.permutation(n)
    n_val = max(1, int(0.1 * n))
    val_idx = set(perm[:n_val].tolist())
    tr_a = [items_a[i] for i in range(n) if i not in val_idx]
    tr_b = [items_b[i] for i in range(n) if i not in val_idx]
    va_a = [items_a[i] for i in sorted(val_idx)]
    va_b = [items_b[i] for i in sorted(val_idx)]
    return tr_a, tr_b, va_a, va_b


# --- Head B (your data) ---
your_imgs, your_masks = build_pair_list(YOUR_IMAGES_DIR, stage_mask_dir)
if your_imgs:
    train_imgs_B, train_masks_B, val_imgs_B, val_masks_B = split_90_10(your_imgs, your_masks, SEED)
    loader_B_train = make_loader(train_imgs_B, train_masks_B, 1, train_tf, True)
    loader_B_val   = make_loader(val_imgs_B,   val_masks_B,   1, val_tf,   False)
    print(f'Head B: {len(train_imgs_B)} train, {len(val_imgs_B)} val')
else:
    loader_B_train = loader_B_val = None
    val_imgs_B = []
    print('⚠️ No paired Head B images found — check Cell 5 output')

# --- Head A (Coralscapes) ---
if HAS_CORALSCAPES:
    cor_imgs, cor_masks = build_pair_list(CORALSCAPES_IMAGES, CORALSCAPES_MASKS)
    if cor_imgs:
        train_imgs_A, train_masks_A, val_imgs_A, val_masks_A = split_90_10(cor_imgs, cor_masks, SEED)
        loader_A_train = make_loader(train_imgs_A, train_masks_A, 0, train_tf, True)
        loader_A_val   = make_loader(val_imgs_A,   val_masks_A,   0, val_tf,   False)
        print(f'Head A: {len(train_imgs_A)} train, {len(val_imgs_A)} val')
    else:
        loader_A_train = loader_A_val = None
        print('⚠️ No Coralscapes pairs found — disabling Head A (single-head mode)')
        HAS_CORALSCAPES = False
else:
    loader_A_train = loader_A_val = None

In [ ]:
# ===========================================================================
# CELL 7: Build the 2-head SegFormer model
# ===========================================================================
# Architecture:
#   Shared encoder (MiT-B2 backbone)  ←  from pretrained Coralscapes model
#   Head A: decode head for 39 Coralscapes classes  ←  pretrained weights
#   Head B: decode head for your classes            ←  fresh classifier,
#                                                      MLP fusion layers warm-copied
#
# forward(x, dataset_id) routes to the correct head. Only one head is used per
# batch (gradient flows through encoder + that head).
#
# RESUME LOGIC: if Cell 2's _resolve_resume_path() finds a previous-stage
# checkpoint, we load shape-compatible weights only. Classifier layers get
# reinitialized if class counts changed (e.g. you dropped Unknown or switched
# KEEP_TOP_N_CLASSES between runs) — that's correct; the encoder transfer is
# what matters and the head learns quickly from the dense SAM labels.
# ===========================================================================

PRETRAINED_CORALSCAPES = "EPFL-ECEO/segformer-b2-finetuned-coralscapes-1024-1024"


class TwoHeadSegFormer(nn.Module):
    def __init__(self, base_model_name, pretrained_coralscapes,
                 num_classes_a, num_classes_b, use_pretrained_coralscapes=True):
        super().__init__()

        if use_pretrained_coralscapes and pretrained_coralscapes:
            print(f'Loading pretrained Coralscapes model: {pretrained_coralscapes}')
            pretrained = SegformerForSemanticSegmentation.from_pretrained(pretrained_coralscapes)
            self.encoder = pretrained.segformer

            actual_n_a = pretrained.config.num_labels
            if actual_n_a != num_classes_a:
                print(f'  Head A num_classes adjusted: {num_classes_a} → {actual_n_a}')
                num_classes_a = actual_n_a

            cfg_a = SegformerConfig.from_pretrained(pretrained_coralscapes)
            self.head_a = SegformerDecodeHead(cfg_a)
            self.head_a.load_state_dict(pretrained.decode_head.state_dict())

            cfg_b = SegformerConfig.from_pretrained(base_model_name, num_labels=num_classes_b)
            self.head_b = SegformerDecodeHead(cfg_b)
            sd = {k: v for k, v in pretrained.decode_head.state_dict().items()
                  if 'classifier' not in k}
            self.head_b.load_state_dict(sd, strict=False)
            print('  Head B: decoder MLPs warm-copied, classifier fresh')
        else:
            print(f'Loading base model (ImageNet init only): {base_model_name}')
            base = SegformerForSemanticSegmentation.from_pretrained(
                base_model_name, num_labels=num_classes_b, ignore_mismatched_sizes=True)
            self.encoder = base.segformer
            cfg_a = SegformerConfig.from_pretrained(base_model_name, num_labels=num_classes_a)
            cfg_b = SegformerConfig.from_pretrained(base_model_name, num_labels=num_classes_b)
            self.head_a = SegformerDecodeHead(cfg_a)
            self.head_b = SegformerDecodeHead(cfg_b)
            self.head_b.load_state_dict(base.decode_head.state_dict(), strict=False)

        self.num_classes_a = num_classes_a
        self.num_classes_b = num_classes_b

    def forward(self, pixel_values, dataset_id):
        features = self.encoder(pixel_values, output_hidden_states=True, return_dict=True).hidden_states
        head = self.head_a if dataset_id == 0 else self.head_b
        logits = head(features)
        return F.interpolate(logits, size=pixel_values.shape[-2:],
                              mode='bilinear', align_corners=False)


use_pretrained = USE_PRETRAINED_CORAL and ('mit-b2' in MODEL_NAME)
if USE_PRETRAINED_CORAL and 'mit-b2' not in MODEL_NAME:
    print(f'ℹ️ {MODEL_NAME} ≠ mit-b2 → pretrained Coralscapes skipped (ImageNet init)')

model = TwoHeadSegFormer(
    base_model_name=MODEL_NAME,
    pretrained_coralscapes=PRETRAINED_CORALSCAPES,
    num_classes_a=N_CLASSES_A,
    num_classes_b=N_CLASSES_B,
    use_pretrained_coralscapes=use_pretrained,
).to(device)

# Sync Head A class list if the pretrained model had a different count
if use_pretrained and model.num_classes_a != N_CLASSES_A:
    N_CLASSES_A = model.num_classes_a
    PALETTE_A = make_palette(N_CLASSES_A, seed=1)
    if len(CORALSCAPES_CLASSES) < N_CLASSES_A:
        CORALSCAPES_CLASSES = CORALSCAPES_CLASSES + [
            f'class_{i}' for i in range(len(CORALSCAPES_CLASSES), N_CLASSES_A)]
    CORALSCAPES_CLASSES = CORALSCAPES_CLASSES[:N_CLASSES_A]

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\nModel: {MODEL_NAME} two-head — {n_params:.1f}M params')
print(f'  Head A: {model.num_classes_a} classes — {"pretrained Coralscapes" if use_pretrained else "scratch"}')
print(f'  Head B: {model.num_classes_b} classes — fresh')

# --- Resume from previous experiment / stage (shape-filtered) -------------
_resume = _resolve_resume_path()
if _resume:
    sd = torch.load(_resume, map_location=device)
    prev_state = sd.get('model', sd)
    cur_state  = model.state_dict()
    # Keep only keys that exist in both AND have matching shapes.
    # This lets us switch class counts between runs (drop Unknown, change top-N)
    # without crashing — the affected classifier layer stays freshly initialized.
    compatible  = {k: v for k, v in prev_state.items()
                   if k in cur_state and cur_state[k].shape == v.shape}
    mismatched  = [k for k, v in prev_state.items()
                   if k in cur_state and cur_state[k].shape != v.shape]
    model.load_state_dict(compatible, strict=False)
    print(f'\n✅ Resumed from {_resume}')
    print(f'   Loaded {len(compatible)}/{len(prev_state)} tensors '
          f'(mIoU of that ckpt: {sd.get("miou", -1):.4f})')
    if mismatched:
        print(f'   ⓘ {len(mismatched)} tensors reinitialized due to shape change '
              f'(e.g. classifier head — expected if class count changed):')
        for k in mismatched[:5]:
            print(f'      {k}  prev={tuple(prev_state[k].shape)}  new={tuple(cur_state[k].shape)}')
else:
    print(f'\nℹ️ No resume checkpoint found — using pretrained Coralscapes init.')
    if RESUME_FROM_STAGE:
        print(f'   (Looked for experiment matching stage "{RESUME_FROM_STAGE}" '
              f'and legacy {CKPT_DIR}/{RESUME_FROM_STAGE}_best.pt)')

---

## 🏋️ Training

The training cell below is the long one (20-40 min on L4). It:

- Trains for `NUM_EPOCHS` = 20 by default
- Each batch randomly picks Head A (Coralscapes) or Head B (your data); probability of Head A decays from 0.5 → 0.2 over epochs
- Saves `{STAGE}_best.pt` to Drive after every epoch that improves `val_mIoU_B`
- Skippable if you just want to re-evaluate: if `{STAGE}_best.pt` already exists, the validation cells will reload it

**Watch for:**
- `mIoU_A` should stay stable (Head A is anchored by pretrained weights)
- `mIoU_B` climbs slowly. In `points` stage it will be low (~0.1-0.3) — that's normal because val targets are single pixels. **Trust the patch accuracy cell more.**
- If loss goes NaN → lower `LR` to 3e-5 in Cell 2, restart.

In [ ]:
# ===========================================================================
# CELL 8: Training loop — joint 2-head sampling
# ===========================================================================
# Each batch comes from EXACTLY ONE dataset (all samples share the same head).
# Per batch, we flip a coin:
#   - prob = CORALSCAPES_WEIGHT_START → END (linearly decays over epochs)
#     → Head A (Coralscapes regularizer)
#   - otherwise → Head B (your data — this is what you deploy)
#
# Loss: cross-entropy with ignore_index=255 (unlabeled / padded / dropped-
# class pixels are skipped). The encoder updates from both heads.
#
# Saves (all inside {EXPERIMENT_DIR}/):
#   best.pt       — highest val mIoU_B so far (used for resuming and bundle)
#   final.pt      — last-epoch weights
#   history.json  — per-epoch loss + mIoU (also written live, survives disconnect)

def compute_iou(conf):
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    denom = tp + fp + fn
    return np.where(denom > 0, tp / (denom + 1e-9), np.nan)


@torch.no_grad()
def evaluate(model, loader, dataset_id, num_classes):
    if loader is None: return None
    model.eval()
    conf = np.zeros((num_classes, num_classes), dtype=np.int64)
    for imgs, masks, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.numpy()
        with _autocast(USE_AMP):
            logits = model(imgs, dataset_id)
        pred = logits.argmax(1).cpu().numpy()
        valid = masks != IGNORE_INDEX
        m = masks[valid]; p = pred[valid]
        conf += np.bincount(m * num_classes + p,
                            minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return {'miou': float(np.nanmean(compute_iou(conf)))}


def dice_loss(logits, targets, ignore_index=255, smooth=1.0):
    """Soft Dice loss — stable for sparse point labels (most pixels ignored)."""
    import torch.nn.functional as _F
    n_cls = logits.shape[1]
    mask  = targets != ignore_index
    if mask.sum() == 0:
        return logits.sum() * 0  # no labeled pixels → zero loss, keep graph
    t = targets.clone(); t[~mask] = 0
    oh   = _F.one_hot(t, n_cls).permute(0,3,1,2).float()  # (B,C,H,W)
    prob = logits.softmax(dim=1)
    m    = mask.unsqueeze(1).float()
    inter = (prob * oh * m).sum(dim=(0,2,3))
    denom = ((prob + oh) * m).sum(dim=(0,2,3))
    return 1 - ((2*inter + smooth) / (denom + smooth)).mean()

def dice_ce_loss(logits, targets, ignore_index=255, ce_weight=0.5, dice_weight=0.5):
    """
    Combined Dice + Cross-Entropy loss.

    Why this beats plain CE for stage 4:
      - CE alone rewards over-prediction of dominant classes (the "smoothing shortcut"
        that gives high mIoU but blurs edges).
      - Dice penalizes over-prediction proportionally — predicting the dominant class
        on extra pixels grows union faster than intersection, hurting Dice.
      - The combo gets stable CE gradients + Dice's anti-shortcut behavior.

    Default 0.5/0.5 weighting is a common starting point in segmentation literature
    (medical imaging, satellite, benthic). Adjust via dice_weight / ce_weight if needed.
    """
    import torch.nn.functional as _F
    ce  = _F.cross_entropy(logits, targets, ignore_index=ignore_index)
    dl  = dice_loss(logits, targets, ignore_index=ignore_index)
    return ce_weight * ce + dice_weight * dl


def focal_ce_loss(logits, targets, ignore_index=255, gamma=2.0):
    """
    Focal Cross-Entropy: down-weights easy examples, focuses on hard ones (often boundaries).
    Equivalent to standard CE when gamma=0.
    """
    import torch.nn.functional as _F
    ce  = _F.cross_entropy(logits, targets, ignore_index=ignore_index, reduction='none')
    pt  = torch.exp(-ce)  # prob of correct class
    return ((1 - pt) ** gamma * ce).mean()


def train_one_epoch(model, loader_A, loader_B, optimizer, scaler, loss_fn,
                     coralscapes_prob, epoch, total_epochs):
    model.train()
    iter_A = iter(loader_A) if loader_A else None
    iter_B = iter(loader_B) if loader_B else None
    n_total = (len(loader_A) if loader_A else 0) + (len(loader_B) if loader_B else 0)
    if n_total == 0:
        raise RuntimeError('No training loaders available')

    running = 0.0; seen = 0; skipped_nan = 0
    for step in range(n_total):
        if iter_A is None:   use_A = False
        elif iter_B is None: use_A = True
        else:                use_A = (random.random() < coralscapes_prob)

        try:
            batch = next(iter_A if use_A else iter_B)
        except StopIteration:
            if use_A: iter_A = iter(loader_A); batch = next(iter_A)
            else:     iter_B = iter(loader_B); batch = next(iter_B)

        imgs, masks, _ = batch
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        dataset_id = 0 if use_A else 1

        if (masks != IGNORE_INDEX).sum() == 0:
            skipped_nan += 1
            continue

        optimizer.zero_grad(set_to_none=True)
        with _autocast(USE_AMP):
            logits = model(imgs, dataset_id)
            # Stage-aware loss selection.
            # Stage 2 (CSV points): Dice handles 99%-ignore masks robustly.
            # Stages 3/4 (dense masks): user-configurable via STAGE4_LOSS.
            #   "ce"        — plain cross-entropy (original behavior, may over-smooth)
            #   "dice_ce"   — recommended: prevents over-prediction shortcut
            #   "focal_ce"  — focuses on hard pixels (boundaries)
            #   "dice"      — Dice only
            _cs = globals().get("_current_stage")
            if _cs == "2":
                loss = dice_loss(logits, masks, ignore_index=IGNORE_INDEX)
            else:
                _s4_loss = globals().get("STAGE4_LOSS", "ce")
                if _s4_loss == "dice_ce":
                    loss = dice_ce_loss(logits, masks, ignore_index=IGNORE_INDEX)
                elif _s4_loss == "focal_ce":
                    loss = focal_ce_loss(logits, masks, ignore_index=IGNORE_INDEX)
                elif _s4_loss == "dice":
                    loss = dice_loss(logits, masks, ignore_index=IGNORE_INDEX)
                else:  # "ce" or anything else → fallback
                    loss = loss_fn(logits, masks)

        if not torch.isfinite(loss):
            skipped_nan += 1
            continue

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item(); seen += 1
        if (step + 1) % 50 == 0:
            print(f'  epoch {epoch+1}/{total_epochs} step {step+1}/{n_total} '
                  f'loss={running/seen:.4f}')
    if skipped_nan:
        print(f'  (skipped {skipped_nan} invalid batches this epoch)')
    return running / max(seen, 1)


# --- Optimizer + loss ---
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler("cuda", enabled=USE_AMP)
loss_fn   = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

active_A_train = loader_A_train if HAS_CORALSCAPES else None
active_A_val   = loader_A_val   if HAS_CORALSCAPES else None
active_B_train = loader_B_train
active_B_val   = loader_B_val

# --- Output paths (inside the experiment folder) ---
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
BEST_CKPT    = os.path.join(EXPERIMENT_DIR, 'best.pt')
FINAL_CKPT   = os.path.join(EXPERIMENT_DIR, 'final.pt')
HISTORY_PATH = os.path.join(EXPERIMENT_DIR, 'history.json')

# --- Training loop ---
# Initialize best_miou from existing checkpoint to prevent overwriting a better model
best_miou = -1.0
if os.path.exists(BEST_CKPT):
    try:
        _existing = torch.load(BEST_CKPT, map_location='cpu', weights_only=False)
        best_miou = float(_existing.get('miou', -1.0))
        print(f'  📌 Existing best.pt has mIoU={best_miou:.4f} — new model only saved if it beats this')
    except Exception as _e:
        print(f'  ⚠️  Could not read existing best.pt mIoU ({_e}), starting at -1.0')
history = []

for epoch in range(NUM_EPOCHS):
    if HAS_CORALSCAPES:
        frac = epoch / max(NUM_EPOCHS - 1, 1)
        c_prob = CORALSCAPES_WEIGHT_START + (CORALSCAPES_WEIGHT_END - CORALSCAPES_WEIGHT_START) * frac
    else:
        c_prob = 0.0

    t0 = time.time()
    avg_loss = train_one_epoch(model, active_A_train, active_B_train,
                                 optimizer, scaler, loss_fn, c_prob, epoch, NUM_EPOCHS)
    dur = time.time() - t0

    metrics = {'epoch': epoch + 1, 'train_loss': avg_loss, 'duration_s': dur,
               'coralscapes_prob': c_prob}
    if active_A_val is not None:
        metrics['val_miou_A'] = evaluate(model, active_A_val, 0, N_CLASSES_A)['miou']
    if active_B_val is not None:
        metrics['val_miou_B'] = evaluate(model, active_B_val, 1, N_CLASSES_B)['miou']
    history.append(metrics)

    # Persist history every epoch so it survives disconnects
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f, indent=2)

    miou_a = metrics.get('val_miou_A', float('nan'))
    miou_b = metrics.get('val_miou_B', float('nan'))
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_loss:.4f}  '
          f'mIoU_A={miou_a:.4f}  mIoU_B={miou_b:.4f}  {dur:.0f}s  c_prob={c_prob:.2f}')

    primary = metrics.get('val_miou_B', -1)
    if primary > best_miou:
        best_miou = primary
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch + 1,
            'miou': best_miou,
            'stage': STAGE,
            'experiment_name': EXPERIMENT_NAME,
            'your_classes': YOUR_CLASSES,
            'coralscapes_classes': CORALSCAPES_CLASSES,
        }, BEST_CKPT)
        print(f'  💾 Saved best → {BEST_CKPT}  (mIoU={best_miou:.4f})')

torch.save({'model': model.state_dict(), 'stage': STAGE,
            'experiment_name': EXPERIMENT_NAME, 'history': history}, FINAL_CKPT)
print(f'\nDone. Best mIoU_B={best_miou:.4f}')
print(f'  Best  → {BEST_CKPT}')
print(f'  Final → {FINAL_CKPT}')
print(f'  Next step: run Cell 11 to export a downloadable bundle.')


# ═══════════════════════════════════════════════════════════════════════════
# MULTI-STAGE TRAINING CONTINUATION
# If TRAINING_FLOW has more stages after the first, run them automatically.
# Each stage loads the best checkpoint from the previous stage, re-renders
# masks for the new data source, and fine-tunes the model further.
# ═══════════════════════════════════════════════════════════════════════════

# initial stage = first non-warmstart stage from TRAINING_FLOW (skip "1" since it is just pretrained init)
_initial_stage = next((c for c in globals().get("TRAINING_FLOW", "2") if c != "1"), "2")
globals()["_current_stage"] = _initial_stage
_flow_stages = [s for s in globals().get("TRAINING_FLOW", "12") if s != "1"]
_stage_cfgs  = globals().get("STAGE_CONFIGS", {})

if len(_flow_stages) > 1:
    print(f"\nTraining flow: {' → '.join(_flow_stages)}  ({len(_flow_stages)-1} more stages to go)")

    for _next_stage_digit in _flow_stages[1:]:
        globals()["_current_stage"] = _next_stage_digit
        if _next_stage_digit not in _stage_cfgs:
            print(f"  Skipping stage {_next_stage_digit} — not in STAGE_CONFIGS")
            continue

        _cfg = _stage_cfgs[_next_stage_digit]
        _new_stage   = _cfg["stage"]
        _new_epochs  = _cfg["epochs"]
        _new_lr      = _cfg["lr"]
        _stage_label = _cfg["label"]

        print(f"\n{'='*72}")
        print(f"  STAGE {_next_stage_digit}: {_stage_label}  ({_new_epochs} epochs, lr={_new_lr})")
        print(f"{'='*72}")

        # -- Check data source exists -----------------------------------------
        if _new_stage == "gold_masks":
            _data_path = globals().get("YOUR_GOLD_COCO_PATH", "")
            if not _data_path or not os.path.exists(_data_path):
                print(f"  SKIP: gold_masks requires YOUR_GOLD_COCO_PATH → {_data_path}")
                print("  Set YOUR_GOLD_COCO_PATH in Cell 2 and upload gold_coco.json to Drive.")
                continue
        elif _new_stage == "sam_masks":
            _data_path = globals().get("YOUR_SAM_COCO_PATH", "")
            if not _data_path or not os.path.exists(_data_path):
                print(f"  SKIP: sam_masks requires YOUR_SAM_COCO_PATH → {_data_path}")
                continue
        elif _new_stage == "targeted":
            _data_path = globals().get("YOUR_TARGETED_COCO_PATH", "")
            if not _data_path or not os.path.exists(_data_path):
                print(f"  SKIP: targeted requires YOUR_TARGETED_COCO_PATH → {_data_path}")
                print("  Annotate weak classes in Roboflow, upload targeted_coco.json to Drive.")
                continue

        # -- Load best checkpoint from previous stage -------------------------
        best_ckpt_prev = os.path.join(EXPERIMENT_DIR, "best.pt")
        if os.path.exists(best_ckpt_prev):
            sd = torch.load(best_ckpt_prev, map_location=device)
            # strict=False handles class-count changes between stages
            _missing, _unexpected = model.load_state_dict(sd["model"], strict=False)
            print(f"  Loaded prev best ({sd.get('miou', -1):.4f} mIoU) — "
                  f"missing={len(_missing)} unexpected={len(_unexpected)}")
        else:
            print("  No previous best.pt — continuing from current weights")

        # -- Update global STAGE + experiment name ----------------------------
        STAGE = _new_stage
        EXPERIMENT_NAME = f"{_stage_label}_top{KEEP_TOP_N_CLASSES}_" +                           f"{'merged_' if globals().get('USE_LABEL_MERGE') else ''}" +                           f"{_new_epochs}ep"
        EXPERIMENT_DIR  = os.path.join(EXPERIMENTS_DIR, EXPERIMENT_NAME)
        os.makedirs(EXPERIMENT_DIR, exist_ok=True)
        print(f"  Experiment dir: {EXPERIMENT_DIR}")

        # -- Re-render masks for new stage ------------------------------------
        print(f"  Rendering masks for {_stage_label}...")
        import importlib
        _render_mod = importlib.import_module("__main__")

        if _new_stage in ("sam_masks", "gold_masks", "targeted"):
            _coco_src = _data_path
            _stage_cache = os.path.join(MASKS_CACHE_DIR, _new_stage)
            _fp = _fingerprint(_coco_src, {"stage": _new_stage, "classes": YOUR_CLASSES})
            if not FORCE_RERENDER and _is_cache_valid(_stage_cache, _fp):
                print(f"  Cache valid — skipping render")
            else:
                _cnt = render_coco_to_png(_coco_src, YOUR_IMAGES_DIR, _stage_cache,
                                          YOUR_CLASS_TO_IDX)
                _write_cache_marker(_stage_cache, _fp, _cnt)
                print(f"  Rendered {_cnt} masks")
            MASKS_DIR = _stage_cache
        else:
            _csv_src = YOUR_CSV_PATH
            _stage_cache = os.path.join(MASKS_CACHE_DIR, _new_stage)
            _fp = _fingerprint(_csv_src, {"stage": _new_stage, "classes": YOUR_CLASSES,
                                          "radius": globals().get("POINT_RADIUS", 3)})
            if not FORCE_RERENDER and _is_cache_valid(_stage_cache, _fp):
                print(f"  Cache valid — skipping render")
            else:
                _cnt = render_points_to_png(_csv_src, YOUR_IMAGES_DIR, _stage_cache,
                                            YOUR_CLASS_TO_IDX, label_col=label_col)
                _write_cache_marker(_stage_cache, _fp, _cnt)
                print(f"  Rendered {_cnt} masks")
            MASKS_DIR = _stage_cache

        # -- Rebuild dataset + loaders for new stage --------------------------
        _new_imgs, _new_masks = build_pair_list(YOUR_IMAGES_DIR, _stage_cache)
        if not _new_imgs:
            print(f"  ⚠️  No image/mask pairs found for {_stage_label} — skipping stage")
            continue
        _tr_imgs, _tr_masks, _va_imgs, _va_masks = split_90_10(_new_imgs, _new_masks, SEED)
        loader_B_train = make_loader(_tr_imgs, _tr_masks, 1, train_tf, True)
        loader_B_val   = make_loader(_va_imgs, _va_masks, 1, val_tf,   False)
        print(f"  Dataloaders: {len(_tr_imgs)} train / {len(_va_imgs)} val samples")

        # -- Fine-tune with new LR --------------------------------------------
        optimizer = torch.optim.AdamW(model.parameters(), lr=_new_lr,
                                      weight_decay=WEIGHT_DECAY)
        scaler    = GradScaler("cuda", enabled=USE_AMP)
        history   = []
        # Read existing best mIoU to avoid overwriting on resume
        _stage_best_path = os.path.join(EXPERIMENT_DIR, "best.pt")
        if os.path.exists(_stage_best_path):
            try:
                _exist = torch.load(_stage_best_path, map_location='cpu', weights_only=False)
                best_miou = float(_exist.get('miou', 0.0))
                print(f'  📌 Existing stage best.pt has mIoU={best_miou:.4f} — protecting from downgrade')
            except Exception:
                best_miou = 0.0
        else:
            best_miou = 0.0
        t0_stage  = time.time()

        for epoch in range(_new_epochs):
            c_prob = CORALSCAPES_WEIGHT_START +                      (CORALSCAPES_WEIGHT_END - CORALSCAPES_WEIGHT_START) * (epoch / max(_new_epochs - 1, 1))
            tr_loss = train_one_epoch(model, active_A_train, loader_B_train,
                                      optimizer, scaler, loss_fn, c_prob,
                                      epoch, _new_epochs)
            val_b = evaluate(model, loader_B_val, 1, N_CLASSES_B)
            miou_b = val_b["miou"] if val_b else 0.0

            rec = {"epoch": epoch + 1, "stage": _new_stage, "train_loss": tr_loss,
                   "val_miou_B": miou_b}
            history.append(rec)

            _hist_path = os.path.join(EXPERIMENT_DIR, "history.json")
            with open(_hist_path, "w") as _hf:
                json.dump(history, _hf, indent=2)

            print(f"  Epoch {epoch+1}/{_new_epochs}  loss={tr_loss:.4f}  "
                  f"val_mIoU_B={miou_b:.4f}", end="")

            if miou_b > best_miou:
                best_miou = miou_b
                torch.save({"model": model.state_dict(), "miou": miou_b,
                             "epoch": epoch + 1, "stage": _new_stage,
                             "classes": YOUR_CLASSES},
                            os.path.join(EXPERIMENT_DIR, "best.pt"))
                print("  ← best", end="")
            print()

        elapsed_s = time.time() - t0_stage
        print(f"  Stage {_next_stage_digit} done in {elapsed_s/60:.1f} min. "
              f"Best mIoU_B = {best_miou:.4f}")
        print(f"  Saved to {EXPERIMENT_DIR}/best.pt")
else:
    print("  Single-stage flow complete. Set TRAINING_FLOW to e.g. '124' for multi-stage.")


In [ ]:
# ===========================================================================
# CELL 9: Full validation report — IoU, Dice, Precision, Recall + confusion
# ===========================================================================
# WHAT EACH METRIC MEANS (segmentation literature):
#   IoU      = TP / (TP + FP + FN)           — intersection-over-union per class
#   Dice/F1  = 2*TP / (2*TP + FP + FN)       — softer than IoU; equals 2*IoU/(1+IoU)
#   Precision= TP / (TP + FP)                — when you predict this class, how often are you right
#   Recall   = TP / (TP + FN)                — of the actual pixels of this class, how many did you find
#   mIoU     = mean of per-class IoU         — standard segmentation benchmark (your headline number)
#   FWIoU    = Σ (class_freq * IoU_c)        — mIoU weighted by class frequency, realistic for imbalanced data
#   mRecall  = mean of per-class recall      — \"macro pixel accuracy\"; equal weight to rare and common classes
#   Pixel acc= Σ correct / Σ total           — overall accuracy, heavily dominated by majority classes
#
# ALSO PRINTED:
#   - per-class table sorted by IoU (top 15 + bottom 10 with support > 0)
#   - top-15 confusion pairs (which classes does the model swap?)
#   - confusion-matrix heatmap saved to {EXPERIMENT_DIR}/confusion_matrix.png
#   - full metrics dumped to {EXPERIMENT_DIR}/metrics.json + confusion_matrix.npy
#
# CAVEAT (points stage): ~100 labeled pixels/image → numbers are noisy.
# Trust sam_masks numbers and the Cell 10 visual sanity check.

# Reload best checkpoint from THIS experiment's folder
best_ckpt = os.path.join(EXPERIMENT_DIR, 'best.pt')
if os.path.exists(best_ckpt):
    sd = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(sd['model'])
    print(f'Loaded best: {best_ckpt}  (saved mIoU={sd.get("miou", -1):.4f})')
else:
    print(f'ⓘ No best.pt found in {EXPERIMENT_DIR} — evaluating current model state.')


def _metrics_from_conf(conf):
    """All per-class + summary metrics from a K×K confusion matrix.
    conf[i, j] = number of pixels whose true class is i and predicted class is j.
    """
    conf = conf.astype(np.float64)
    K = conf.shape[0]
    tp = np.diag(conf)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    support = conf.sum(axis=1)           # true-pixel count per class
    total = conf.sum()

    with np.errstate(divide='ignore', invalid='ignore'):
        iou       = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
        dice      = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
        precision = np.where(tp + fp > 0, tp / (tp + fp), np.nan)
        recall    = np.where(tp + fn > 0, tp / (tp + fn), np.nan)

    miou     = float(np.nanmean(iou))
    mdice    = float(np.nanmean(dice))
    mrecall  = float(np.nanmean(recall))
    mprec    = float(np.nanmean(precision))
    pixel_acc = float(tp.sum() / max(total, 1))
    freq = support / max(total, 1)
    fwiou = float(np.nansum(np.where(np.isnan(iou), 0, iou * freq)))

    return {
        'per_class': {
            'iou': iou, 'dice': dice, 'precision': precision,
            'recall': recall, 'support': support.astype(np.int64),
        },
        'summary': {
            'mIoU': miou, 'FWIoU': fwiou, 'mDice_F1': mdice,
            'mRecall_macro': mrecall, 'mPrecision_macro': mprec,
            'pixel_accuracy': pixel_acc,
            'n_classes_with_support': int((support > 0).sum()),
            'n_classes_total': int(K),
        },
    }


def _confusion_pairs(conf, classes, k=15):
    """Top k off-diagonal (true, pred, count) confusions, sorted by pixels mis-routed."""
    c = conf.copy().astype(np.int64)
    np.fill_diagonal(c, 0)
    pairs = []
    idx = np.argsort(-c, axis=None)[:k]
    for flat in idx:
        i, j = np.unravel_index(flat, c.shape)
        n = int(c[i, j])
        if n == 0: break
        pairs.append((classes[i], classes[j], n,
                      float(n / max(conf[i].sum(), 1))))  # fraction of true-i pixels routed to j
    return pairs


@torch.no_grad()
def build_confusion(loader, dataset_id, num_classes):
    model.eval()
    conf = np.zeros((num_classes, num_classes), dtype=np.int64)
    for imgs, masks, _ in loader:
        imgs = imgs.to(device)
        with _autocast(USE_AMP):
            logits = model(imgs, dataset_id)
        pred = logits.argmax(1).cpu().numpy()
        m = masks.numpy()
        valid = m != IGNORE_INDEX
        conf += np.bincount((m[valid] * num_classes + pred[valid]).ravel(),
                             minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    return conf


def _print_report(conf, classes, title, save_prefix=None):
    m = _metrics_from_conf(conf)
    s = m['summary']; pc = m['per_class']

    print('\n' + '═' * 78)
    print(f'  {title}')
    print('═' * 78)
    print(f'  {"mIoU (macro)":<26} {s["mIoU"]:.4f}     ← headline benchmark number')
    print(f'  {"FWIoU (freq-weighted)":<26} {s["FWIoU"]:.4f}     ← realistic for imbalanced data')
    print(f'  {"Mean Dice / F1":<26} {s["mDice_F1"]:.4f}     ← softer than IoU')
    print(f'  {"Mean Recall (macro)":<26} {s["mRecall_macro"]:.4f}     ← avg class-recall, \"macro px acc\"')
    print(f'  {"Mean Precision (macro)":<26} {s["mPrecision_macro"]:.4f}')
    print(f'  {"Pixel accuracy":<26} {s["pixel_accuracy"]:.4f}     ← dominated by frequent classes')
    print(f'  Classes with val support: {s["n_classes_with_support"]} / {s["n_classes_total"]}')

    # Per-class table
    order = np.argsort(-np.nan_to_num(pc['iou'], nan=-1))
    def _row(i):
        return (f'  {classes[i][:26]:<26}  '
                f'IoU={pc["iou"][i]:.3f}  Dice={pc["dice"][i]:.3f}  '
                f'P={pc["precision"][i]:.3f}  R={pc["recall"][i]:.3f}  '
                f'n={int(pc["support"][i])}')
    top = [i for i in order if pc['support'][i] > 0][:15]
    bottom = [i for i in order[::-1] if pc['support'][i] > 0][:10]
    print('\n  ── Top 15 classes by IoU ──')
    for i in top: print(_row(i))
    if len(bottom) > 0 and set(bottom) != set(top):
        print('\n  ── Bottom 10 classes by IoU (with support > 0) ──')
        for i in bottom: print(_row(i))

    # Confusion pairs
    pairs = _confusion_pairs(conf, classes, k=15)
    if pairs:
        print('\n  ── Top 15 confusion pairs (true → pred) ──')
        print(f'  {"true":<22} → {"predicted as":<22}   pixels    % of true')
        for t, p, n, frac in pairs:
            print(f'  {t[:22]:<22} → {p[:22]:<22}  {n:>8}    {frac*100:5.1f}%')

    # Save artefacts
    if save_prefix:
        os.makedirs(EXPERIMENT_DIR, exist_ok=True)
        # Raw confusion matrix
        np.save(os.path.join(EXPERIMENT_DIR, f'{save_prefix}_confusion_matrix.npy'), conf)
        # JSON with all numbers
        out = {
            'summary':  {k: float(v) if isinstance(v, float) else v for k, v in s.items()},
            'per_class': {
                classes[i]: {
                    'iou':       None if np.isnan(pc['iou'][i])       else float(pc['iou'][i]),
                    'dice':      None if np.isnan(pc['dice'][i])      else float(pc['dice'][i]),
                    'precision': None if np.isnan(pc['precision'][i]) else float(pc['precision'][i]),
                    'recall':    None if np.isnan(pc['recall'][i])    else float(pc['recall'][i]),
                    'support':   int(pc['support'][i]),
                }
                for i in range(len(classes))
            },
            'top_confusions': [
                {'true': t, 'predicted_as': p, 'pixels': n, 'frac_of_true': frac}
                for t, p, n, frac in pairs
            ],
        }
        mpath = os.path.join(EXPERIMENT_DIR, f'{save_prefix}_metrics.json')
        with open(mpath, 'w') as f:
            json.dump(out, f, indent=2)
        print(f'\n  💾 Saved metrics → {mpath}')
        print(f'  💾 Saved confusion matrix → {save_prefix}_confusion_matrix.npy')

        # Heatmap — only if the matrix is not too huge
        if len(classes) <= 60:
            fig, ax = plt.subplots(figsize=(max(8, len(classes) * 0.3),
                                             max(7, len(classes) * 0.3)))
            with np.errstate(divide='ignore', invalid='ignore'):
                row_norm = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)
            im = ax.imshow(row_norm, cmap='magma', aspect='auto', vmin=0, vmax=1)
            ax.set_xticks(range(len(classes)));  ax.set_yticks(range(len(classes)))
            ax.set_xticklabels(classes, rotation=90, fontsize=7)
            ax.set_yticklabels(classes, fontsize=7)
            ax.set_xlabel('Predicted'); ax.set_ylabel('True')
            ax.set_title(f'{title} — row-normalized confusion (diag = recall)')
            plt.colorbar(im, ax=ax, fraction=0.03)
            plt.tight_layout()
            hpath = os.path.join(EXPERIMENT_DIR, f'{save_prefix}_confusion_matrix.png')
            plt.savefig(hpath, dpi=120)
            plt.show()
            print(f'  💾 Saved heatmap → {hpath}')

    return m


# ── Run for Head B (your data — what you deploy) ──────────────────────────
if active_B_val is not None:
    conf_B = build_confusion(active_B_val, 1, N_CLASSES_B)
    metrics_B = _print_report(conf_B, YOUR_CLASSES, 'Head B (your data)  —  validation report',
                               save_prefix='headB')
    iou_B = metrics_B['per_class']['iou']
else:
    metrics_B = None

# ── Head A only if joint-training was enabled ─────────────────────────────
if active_A_val is not None:
    conf_A = build_confusion(active_A_val, 0, N_CLASSES_A)
    metrics_A = _print_report(conf_A, CORALSCAPES_CLASSES, 'Head A (Coralscapes)  —  validation report',
                               save_prefix='headA')
    iou_A = metrics_A['per_class']['iou']

# ── Interpretation hints ──────────────────────────────────────────────────
if metrics_B:
    s = metrics_B['summary']
    print('\n' + '─' * 78)
    print('  💡 Interpretation')
    print('─' * 78)
    if s['FWIoU'] > s['mIoU'] + 0.08:
        print("  • FWIoU ≫ mIoU  → you are doing well on common classes but rare classes drag mIoU down.")
        print('    Try KEEP_TOP_N_CLASSES=25 for an even cleaner baseline.')
    if s['mRecall_macro'] > s['mPrecision_macro'] + 0.05:
        print('  • Recall > Precision → model over-predicts some classes. Look at top confusion pairs '
              'to see which ones bleed into others.')
    elif s['mPrecision_macro'] > s['mRecall_macro'] + 0.05:
        print('  • Precision > Recall → model is conservative, misses pixels. Common with sparse labels.')
    print('  • Confusion pairs with >20% routing are real weaknesses — check if those classes are '
          'visually similar in your data (e.g. Por ↔ Acr).')


# ═══════════════════════════════════════════════════════════════════════════
# FULL CONFUSION MATRIX — every class vs. every other class
# Row = true class, Column = predicted class.
# Colour = fraction of true-class pixels predicted as that column class
#          (row-normalised so rare classes are visible alongside common ones).
# ═══════════════════════════════════════════════════════════════════════════
# METRIC LEGEND (printed here so you always have it next to the chart):
#   IoU      = TP/(TP+FP+FN)          Strict overlap. The standard benchmark.
#   Dice/F1  = 2TP/(2TP+FP+FN)        Slightly less harsh than IoU; F1 in ML.
#   Precision= TP/(TP+FP)             When you predict class C, how often correct?
#   Recall   = TP/(TP+FN)             Of all true C pixels, how many did you find?
#   mIoU     = mean(IoU per class)     Headline number — weight rare = common.
#   FWIoU    = Σ freq_c * IoU_c        Like mIoU but weighted by pixel frequency.
#   PixAcc   = Σ correct / Σ total     Easy to inflate with dominant classes.
# ═══════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as _np_cm

# Rebuild confusion on validation set (full matrix, not just summary)
conf_full = build_confusion(loader_B_val, 1, N_CLASSES_B)
classes_b = YOUR_IDX_TO_CLASS  # dict idx→name

n_cls = conf_full.shape[0]
labels = [classes_b.get(i, str(i)) for i in range(n_cls)]

# Row-normalise (fraction of true-class pixels)
row_sums = conf_full.sum(axis=1, keepdims=True).clip(min=1)
conf_norm = conf_full.astype(np.float32) / row_sums

# ── Per-class summary table ─────────────────────────────────────────────
m  = _metrics_from_conf(conf_full)
pc = m["per_class"]
s  = m["summary"]

print("\n" + "═"*78)
print("  PER-CLASS METRICS  (sorted by IoU descending, support > 0 only)")
print("═"*78)
print(f"  {'Class':<20} {'IoU':>7} {'Dice':>7} {'Prec':>7} {'Recall':>7} {'Support':>9}")
print("  " + "-"*62)
_idx_sorted = sorted(
    [i for i in range(n_cls) if pc["support"][i] > 0],
    key=lambda i: (pc["iou"][i] if not np.isnan(pc["iou"][i]) else -1),
    reverse=True,
)
for i in _idx_sorted:
    nm  = labels[i]
    iou = pc["iou"][i];   d = pc["dice"][i]
    pr  = pc["precision"][i]; rc = pc["recall"][i]
    sup = int(pc["support"][i])
    _fmt = lambda v: f"{v:.3f}" if not np.isnan(v) else "  —  "
    print(f"  {nm:<20} {_fmt(iou):>7} {_fmt(d):>7} {_fmt(pr):>7} {_fmt(rc):>7} {sup:>9,}")

classes_no_support = [labels[i] for i in range(n_cls) if pc["support"][i] == 0]
if classes_no_support:
    print(f"\n  Classes with zero validation pixels (not in val set): {classes_no_support}")

print("\n" + "═"*78)
print(f"  SUMMARY")
print(f"  {'mIoU':<26} {s['mIoU']:.4f}   ← headline benchmark")
print(f"  {'FWIoU (freq-weighted)':<26} {s['FWIoU']:.4f}   ← realistic for imbalanced data")
print(f"  {'Pixel accuracy':<26} {s['pixel_accuracy']:.4f}   ← dominated by big classes")
print(f"  {'mDice / macro F1':<26} {s['mDice_F1']:.4f}")
print(f"  {'Macro precision':<26} {s['mPrecision_macro']:.4f}")
print(f"  {'Macro recall':<26} {s['mRecall_macro']:.4f}")
print(f"  {'Classes with support':<26} {s['n_classes_with_support']} / {s['n_classes_total']}")
print("═"*78)

# ── Confusion matrix heatmap ────────────────────────────────────────────
_plot_classes = [labels[i] for i in range(n_cls) if pc["support"][i] > 0]
_plot_idx     = [i for i in range(n_cls) if pc["support"][i] > 0]
_conf_plot    = conf_norm[np.ix_(_plot_idx, _plot_idx)]

fig_sz = max(12, len(_plot_classes) * 0.45)
fig, ax = plt.subplots(figsize=(fig_sz, fig_sz * 0.9))
im = ax.imshow(_conf_plot, cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03, label="Fraction of true-class pixels predicted as column class")

ax.set_xticks(range(len(_plot_classes)))
ax.set_xticklabels(_plot_classes, rotation=90, fontsize=max(7, 11 - len(_plot_classes)//8))
ax.set_yticks(range(len(_plot_classes)))
ax.set_yticklabels(_plot_classes, fontsize=max(7, 11 - len(_plot_classes)//8))
ax.set_xlabel("Predicted class", fontsize=12)
ax.set_ylabel("True class", fontsize=12)
ax.set_title(
    f"Confusion matrix — {EXPERIMENT_NAME}\n"
    f"mIoU={s['mIoU']:.3f}  |  {len(_plot_classes)} classes with val-set support\n"
    f"Row-normalised: diagonal = recall per class",
    fontsize=13, fontweight="bold"
)

# Annotate cells with value if > 0.05 (avoid clutter)
thresh = _conf_plot.max() / 2.0
for i in range(len(_plot_classes)):
    for j in range(len(_plot_classes)):
        v = _conf_plot[i, j]
        if v > 0.05:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=max(5, 9 - len(_plot_classes)//10),
                    color="white" if v > thresh else "black")

plt.tight_layout()
_cm_png = os.path.join(EXPERIMENT_DIR, "confusion_matrix.png")
plt.savefig(_cm_png, dpi=120, bbox_inches="tight")
plt.show()
print(f"  Confusion matrix saved to {_cm_png}")

# Save full metrics JSON
_metrics_out = {
    "experiment": EXPERIMENT_NAME, "stage": STAGE,
    "summary": s,
    "per_class": {
        labels[i]: {
            "iou":       float(pc["iou"][i])       if not np.isnan(pc["iou"][i]) else None,
            "dice":      float(pc["dice"][i])      if not np.isnan(pc["dice"][i]) else None,
            "precision": float(pc["precision"][i]) if not np.isnan(pc["precision"][i]) else None,
            "recall":    float(pc["recall"][i])    if not np.isnan(pc["recall"][i]) else None,
            "support":   int(pc["support"][i]),
        }
        for i in range(n_cls)
    },
}
import json as _json
with open(os.path.join(EXPERIMENT_DIR, "metrics_full.json"), "w") as _mf:
    _json.dump(_metrics_out, _mf, indent=2)
print(f"  Full metrics saved to {EXPERIMENT_DIR}/metrics_full.json")

# Top confusion pairs (most common misclassifications)
pairs = _confusion_pairs(conf_full, labels, k=20)
print("\n  TOP MISCLASSIFICATIONS (true → predicted, pixel count, % of true class):")
for true_c, pred_c, count, frac in pairs:
    bar = "█" * int(frac * 30)
    print(f"  {true_c:<20} → {pred_c:<20}  {count:>8,}px  {frac*100:5.1f}%  {bar}")


# ═══════════════════════════════════════════════════════════════════════════
# CoralNet-equivalent point accuracy on Ewout's CSV
# Same metric CoralNet reports (% of labeled points predicted correctly)
# This makes a direct apples-to-apples comparison possible.
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*78)
print("  CORALNET-EQUIVALENT POINT ACCURACY  (on YOUR_CSV_PATH points)")
print("═"*78)

import pandas as _pd
from collections import Counter as _Counter

if not os.path.exists(YOUR_CSV_PATH):
    print(f"  CSV not found ({YOUR_CSV_PATH}) — skipping point accuracy.")
else:
    _df = _pd.read_csv(YOUR_CSV_PATH, low_memory=False)
    _label_col = "Label code" if "Label code" in _df.columns else "Label"
    _df = _df.rename(columns={_label_col: "_label"})

    # Apply same merge + filter as training
    if globals().get("USE_LABEL_MERGE", False):
        _df["_label"] = _df["_label"].map(apply_label_merge)
        _df = _df[_df["_label"].notna()]
    _df = _df[_df["_label"].isin(YOUR_CLASS_TO_IDX.keys())]
    _df = _df[["Name", "Row", "Column", "_label"]].dropna()

    if len(_df) == 0:
        print("  No CSV points map to trained classes — skipping.")
    else:
        # Build image index for resolver
        _idx_pa = build_image_index(YOUR_IMAGES_DIR)

        model.eval()
        _img_norm = A.Normalize(mean=(0.485, 0.456, 0.406),
                                std=(0.229, 0.224, 0.225))
        _correct = 0
        _total   = 0
        _per_class_correct = _Counter()
        _per_class_total   = _Counter()
        _missing_imgs = 0

        # Group points by image — one forward pass per image
        _grouped = _df.groupby("Name")
        _n_imgs = len(_grouped)
        print(f"  Evaluating {len(_df):,} points across {_n_imgs} images...")

        with torch.no_grad():
            for _i, (_name, _group) in enumerate(_grouped):
                _img_path = resolve_image(_name, YOUR_IMAGES_DIR, _idx_pa)
                if not _img_path:
                    _missing_imgs += 1
                    continue
                _img = cv2.imread(_img_path)
                if _img is None:
                    _missing_imgs += 1; continue
                _img = cv2.cvtColor(_img, cv2.COLOR_BGR2RGB)
                _h0, _w0 = _img.shape[:2]

                # Resize image to INPUT_SIZE while keeping aspect ratio,
                # then pad — same as val transform
                _scale = INPUT_SIZE / max(_h0, _w0)
                _new_h = int(round(_h0 * _scale)); _new_w = int(round(_w0 * _scale))
                _resized = cv2.resize(_img, (_new_w, _new_h), interpolation=cv2.INTER_AREA)
                _pad_h = INPUT_SIZE - _new_h
                _pad_w = INPUT_SIZE - _new_w
                _padded = cv2.copyMakeBorder(_resized, 0, _pad_h, 0, _pad_w,
                                             cv2.BORDER_CONSTANT, value=0)
                _norm = _img_norm(image=_padded)["image"]
                _tensor = torch.from_numpy(_norm.transpose(2,0,1)).unsqueeze(0).float().to(device)

                with _autocast(USE_AMP):
                    _logits = model(_tensor, dataset_id=1)
                # logits: (1, C, H, W) at INPUT_SIZE/4 — upsample to INPUT_SIZE
                _logits_up = F.interpolate(_logits, size=(INPUT_SIZE, INPUT_SIZE),
                                           mode="bilinear", align_corners=False)
                _pred = _logits_up.argmax(dim=1)[0].cpu().numpy()  # (INPUT_SIZE,INPUT_SIZE)

                # For each point: map (col_orig, row_orig) → (x_input, y_input)
                for _, _r in _group.iterrows():
                    _x_orig = float(_r["Column"]); _y_orig = float(_r["Row"])
                    _x_in = int(round(_x_orig * _scale))
                    _y_in = int(round(_y_orig * _scale))
                    if 0 <= _x_in < INPUT_SIZE and 0 <= _y_in < INPUT_SIZE:
                        _pred_idx = int(_pred[_y_in, _x_in])
                        _true_idx = YOUR_CLASS_TO_IDX[_r["_label"]]
                        _per_class_total[_r["_label"]] += 1
                        _total += 1
                        if _pred_idx == _true_idx:
                            _correct += 1
                            _per_class_correct[_r["_label"]] += 1

        if _total == 0:
            print("  No points evaluated (all points outside crop or images missing).")
        else:
            _point_acc = _correct / _total
            print(f"\n  📊 POINT ACCURACY: {_point_acc*100:.2f}%  ({_correct:,} / {_total:,} points)")
            print(f"     Missing images: {_missing_imgs}")
            print(f"\n  Compare to CoralNet baseline: 72%   (target to beat)")
            if _point_acc >= 0.72:
                print(f"  🎯 BEAT CORALNET by {(_point_acc-0.72)*100:+.1f} percentage points!")
            else:
                print(f"  Below CoralNet by {(0.72-_point_acc)*100:.1f} percentage points")

            # Per-class point accuracy (top 10 by support)
            _by_support = sorted(_per_class_total.items(), key=lambda x: -x[1])[:15]
            print(f"\n  Per-class point accuracy (top 15 by support):")
            print(f"  {'Class':<20} {'Acc':>8} {'Correct/Total':>16}")
            print("  " + "-"*48)
            for _cls, _tot in _by_support:
                _cor = _per_class_correct.get(_cls, 0)
                _acc = _cor / _tot if _tot else 0
                print(f"  {_cls:<20} {_acc*100:>7.1f}% {_cor:>7,}/{_tot:<7,}")

            # Save to metrics JSON
            _metrics_out["point_accuracy"] = {
                "overall": float(_point_acc),
                "correct": int(_correct),
                "total":   int(_total),
                "missing_images": int(_missing_imgs),
                "coralnet_baseline": 0.72,
                "delta_vs_coralnet": float(_point_acc - 0.72),
                "per_class": {
                    cls: {"correct": int(_per_class_correct.get(cls, 0)),
                          "total":   int(tot),
                          "accuracy": float(_per_class_correct.get(cls, 0) / tot) if tot else None}
                    for cls, tot in _per_class_total.items()
                },
            }
            with open(os.path.join(EXPERIMENT_DIR, "metrics_full.json"), "w") as _mf:
                _json.dump(_metrics_out, _mf, indent=2)
            print(f"\n  Point accuracy saved to {EXPERIMENT_DIR}/metrics_full.json")

print("═"*78)


In [ ]:
# ===========================================================================
# CELL 9b: Export HTML report for sharing (e.g. with your biologist)
# ===========================================================================
# Generates a single .html file you can email or download from Drive.
# Run this after Cell 9. Reads metrics_B and confusion_matrix.png from
# EXPERIMENT_DIR. Output: {EXPERIMENT_DIR}/report_ewout.html
#                         {DRIVE_DIR}/report_{EXPERIMENT_NAME}.html

import base64, io, os, json as _rj, shutil
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

def _fig_to_b64(fig, dpi=110):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# Load metrics JSON
_mpath = os.path.join(EXPERIMENT_DIR, "headB_metrics.json")
if not os.path.exists(_mpath):
    _mpath = os.path.join(EXPERIMENT_DIR, "metrics_full.json")
if not os.path.exists(_mpath):
    print(f"Metrics JSON not found at {_mpath}. Run Cell 9 first.")
    raise SystemExit(0)
with open(_mpath) as _f:
    _m = _rj.load(_f)

_s = _m["summary"]
_pc = _m.get("per_class", {})
_confusions = _m.get("top_confusions", [])

# Per-class IoU bar chart
_cls_data = sorted(
    [(nm, d) for nm, d in _pc.items() if d.get("support", 0) > 0],
    key=lambda x: (x[1]["iou"] or 0), reverse=True
)
_names  = [x[0] for x in _cls_data]
_ious   = [(x[1]["iou"]       or 0) for x in _cls_data]
_precs  = [(x[1]["precision"] or 0) for x in _cls_data]
_recs   = [(x[1]["recall"]    or 0) for x in _cls_data]

fig_bar, ax_bar = plt.subplots(figsize=(max(10, len(_names)*0.38), 5))
_x = np.arange(len(_names)); _w = 0.28
ax_bar.bar(_x - _w, _ious,  _w, label="IoU",       color="#2196F3", alpha=0.9)
ax_bar.bar(_x,      _precs, _w, label="Precision", color="#4CAF50", alpha=0.9)
ax_bar.bar(_x + _w, _recs,  _w, label="Recall",    color="#FF9800", alpha=0.9)
ax_bar.axhline(_s["mIoU"], color="#2196F3", linestyle="--", linewidth=1.2, alpha=0.6,
               label=f'mIoU={_s["mIoU"]:.3f}')
ax_bar.set_xticks(_x)
ax_bar.set_xticklabels(_names, rotation=55, ha="right", fontsize=max(7, 11-len(_names)//8))
ax_bar.set_ylim(0, 1.05)
ax_bar.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax_bar.set_title(f"Per-class IoU, Precision, Recall — {EXPERIMENT_NAME}", fontsize=13)
ax_bar.legend(fontsize=10); ax_bar.grid(axis="y", alpha=0.3)
plt.tight_layout()
_bar_b64 = _fig_to_b64(fig_bar)
plt.close(fig_bar)

# Load confusion matrix PNG
_cm_b64 = None
for _cp in [os.path.join(EXPERIMENT_DIR, "headB_confusion_matrix.png"),
            os.path.join(EXPERIMENT_DIR, "confusion_matrix.png")]:
    if os.path.exists(_cp):
        with open(_cp, "rb") as _f:
            _cm_b64 = base64.b64encode(_f.read()).decode()
        break

# Build table rows
def _pct(v): return "—" if v is None else f"{v*100:.1f}%"
def _bar(v, w=20):
    if v is None: return ""
    return "█" * int((v or 0) * w) + "░" * (w - int((v or 0) * w))

_table_rows = []
for nm, d in _cls_data:
    iou = d.get("iou"); prec = d.get("precision"); rec = d.get("recall")
    dice = d.get("dice"); sup = d.get("support", 0)
    bg = ("background:#e8f5e9" if (iou or 0) >= 0.65
          else "background:#fff9c4" if (iou or 0) >= 0.40
          else "background:#ffebee") if iou is not None else ""
    _table_rows.append(
        f'<tr style="{bg}"><td><b>{nm}</b></td>'
        f'<td>{_pct(iou)}</td><td>{_pct(dice)}</td>'
        f'<td>{_pct(prec)}</td><td>{_pct(rec)}</td>'
        f'<td>{sup:,}</td>'
        f'<td style="font-family:monospace;font-size:11px;color:#555">{_bar(iou)}</td></tr>'
    )
_table_rows_html = "\n".join(_table_rows)

_conf_rows = []
for _pair in _confusions[:20]:
    t = _pair["true"]; p = _pair["predicted_as"]
    frac = _pair.get("frac_of_true", 0); px = _pair.get("pixels", 0)
    col = "#ffebee" if frac > 0.20 else ("#fff9c4" if frac > 0.10 else "")
    _conf_rows.append(
        f'<tr style="background:{col}"><td><b>{t}</b></td><td>→</td>'
        f'<td><b>{p}</b></td><td>{px:,} px</td><td>{frac*100:.1f}%</td></tr>'
    )
_conf_rows_html = "\n".join(_conf_rows)

# Guidance
_miou  = _s.get("mIoU", 0); _fwiou = _s.get("FWIoU", 0)
_pxacc = _s.get("pixel_accuracy", 0); _mdice = _s.get("mDice_F1", 0)
_mrec  = _s.get("mRecall_macro", 0);  _mprec = _s.get("mPrecision_macro", 0)
_ncls  = _s.get("n_classes_with_support", "?"); _ntot = _s.get("n_classes_total", "?")
_stage_label = {"points": "Stage 2 — sparse CSV point annotations",
                "sam_masks": "Stage 3 — SAM polygon masks",
                "gold_masks": "Stage 4 — biologist COCO annotations"}.get(STAGE, STAGE)
_merge_note = "Yes — 94 CoralNet labels merged to ~49 classes" if USE_LABEL_MERGE else "No — original labels"

_guidance = []
if _miou < 0.35:
    _guidance.append(("⚠️ mIoU &lt; 35%",
        "Low overall accuracy — common for Stage 2 (point annotations). "
        "Retrain with SAM masks or gold annotations for a meaningful CoralNet comparison."))
elif _miou < 0.55:
    _guidance.append(("🔶 mIoU 35–55%",
        "Moderate accuracy. Competitive for dense segmentation. "
        "CoralNet point-classification typically reports 60–80% accuracy on its own label set, "
        "but CoralNet only labels sparse points — full segmentation is a harder task."))
else:
    _guidance.append(("✅ mIoU &gt; 55%",
        "Good accuracy for full segmentation. "
        "This is significantly harder than CoralNet point-classification, "
        "so direct numerical comparison should account for that."))
if _fwiou > _miou + 0.08:
    _guidance.append(("📊 FWIoU ≫ mIoU",
        "Doing well on common classes but rare classes drag mIoU down. "
        "Check which rare classes have low IoU in the per-class table."))
if _mrec > _mprec + 0.07:
    _guidance.append(("🔵 Recall &gt; Precision",
        "Model tends to over-predict classes. Common with sparse training data."))
elif _mprec > _mrec + 0.07:
    _guidance.append(("🔵 Precision &gt; Recall",
        "Model is conservative — misses some pixels. Common with point-annotation training."))
_guidance_html = "\n".join(
    f'<div class="guidance"><b>{t}</b> — {txt}</div>' for t, txt in _guidance
)

# Confusion matrix section
_cm_section = ""
if _cm_b64:
    _cm_section = (
        "<h2>Confusion Matrix</h2>"
        '<p class="note">Rows = true class, columns = predicted class. '
        "Colour = fraction of that class's pixels predicted as each column. "
        "The diagonal is recall per class — you want it bright. "
        "Off-diagonal bright cells = common mix-ups.</p>"
        f'<img src="data:image/png;base64,{_cm_b64}" '
        'style="max-width:100%;border:1px solid #ddd;border-radius:6px">'
    )

# Assemble HTML
_css = """
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
     max-width:1100px;margin:40px auto;padding:0 20px;color:#222}
h1{font-size:1.6em;border-bottom:3px solid #2196F3;padding-bottom:8px}
h2{font-size:1.2em;margin-top:36px;color:#1565C0}
table{border-collapse:collapse;width:100%;margin:12px 0;font-size:.9em}
th{background:#1565C0;color:#fff;padding:8px 10px;text-align:left}
td{padding:6px 10px;border-bottom:1px solid #eee}
.meta{display:flex;gap:30px;flex-wrap:wrap;background:#f5f5f5;
      padding:14px 18px;border-radius:8px;margin:16px 0;font-size:.9em}
.meta div{flex:1;min-width:200px}
.meta b{display:block;color:#555;font-size:.8em;text-transform:uppercase;letter-spacing:.05em}
.summary{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin:12px 0}
.metric{background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:12px 16px}
.metric .val{font-size:2em;font-weight:700;color:#1565C0}
.metric .lbl{font-size:.8em;font-weight:600;text-transform:uppercase;color:#666;margin-bottom:4px}
.metric .def{font-size:.78em;color:#888;margin-top:4px}
.guidance{background:#e3f2fd;border-left:4px solid #2196F3;padding:10px 14px;
          margin:8px 0;border-radius:0 6px 6px 0;font-size:.88em}
.note{font-size:.82em;color:#555;font-style:italic;margin:4px 0 10px}
img{display:block;margin:0 auto}
footer{margin-top:48px;font-size:.75em;color:#aaa;border-top:1px solid #eee;padding-top:12px}
"""

# ── Add metrics-explanation HTML block to the report ─────────────────────────
_metrics_explanation_html = """
<section style='background:#f4f6f8;padding:18px 24px;border-radius:8px;margin:24px 0;'>
  <h2 style='margin-top:0;color:#1a3a52;'>📐 How to read these metrics</h2>

  <p><strong>Two different metrics are reported below.</strong> They measure different things, and CoralNet only reports one of them.</p>

  <h3 style='color:#2a5a82;margin-bottom:6px;'>1. Point accuracy (= CoralNet's metric)</h3>
  <p>For each labeled point in the test images, the model predicts a class. <em>Point accuracy</em> = % of points predicted correctly. This is exactly the metric CoralNet uses, and is directly comparable to their reported 72%.</p>

  <h3 style='color:#2a5a82;margin-bottom:6px;'>2. mIoU (mean Intersection over Union)</h3>
  <p>For each class, IoU = overlap area between prediction and ground truth, divided by the union. mIoU averages this across all classes. <strong>It is much harsher than accuracy</strong> because:</p>
  <ul style='margin-top:4px;'>
    <li>It penalizes both false positives and false negatives</li>
    <li>Rare classes weigh equally in the mean as common ones</li>
    <li>It measures spatial extent, not just class identity at a point</li>
  </ul>

  <h3 style='color:#2a5a82;margin-bottom:6px;'>Conversion (rule of thumb)</h3>
  <table style='border-collapse:collapse;margin-top:8px;font-size:14px;'>
    <thead style='background:#dde6ef;'>
      <tr><th style='padding:6px 12px;text-align:left;'>mIoU</th><th style='padding:6px 12px;text-align:left;'>Pixel accuracy</th><th style='padding:6px 12px;text-align:left;'>Point accuracy</th></tr>
    </thead>
    <tbody>
      <tr><td style='padding:4px 12px;'>0.30</td><td style='padding:4px 12px;'>~82%</td><td style='padding:4px 12px;'>~70–75%</td></tr>
      <tr><td style='padding:4px 12px;'>0.40</td><td style='padding:4px 12px;'>~87%</td><td style='padding:4px 12px;'>~78–82%</td></tr>
      <tr><td style='padding:4px 12px;'>0.50</td><td style='padding:4px 12px;'>~90%</td><td style='padding:4px 12px;'>~83–87%</td></tr>
      <tr><td style='padding:4px 12px;'>0.65</td><td style='padding:4px 12px;'>~94%</td><td style='padding:4px 12px;'>~88–93%</td></tr>
    </tbody>
  </table>

  <h3 style='color:#2a5a82;margin-bottom:6px;'>Why segmentation > point classification</h3>
  <p>Even at equal point accuracy, this segmentation model gives Ewout something CoralNet fundamentally cannot:</p>
  <ul style='margin-top:4px;'>
    <li><strong>Coral colony boundaries</strong> — for size and area estimation</li>
    <li><strong>Coverage % per class</strong> — the key ecological metric</li>
    <li><strong>Colony counts</strong> and individual shapes</li>
    <li><strong>Within-colony zones</strong> — bleaching, dead vs. live</li>
  </ul>
  <p style='margin-bottom:0;'>A CoralNet model with 95% accuracy still cannot tell you the area of a brain coral. This SegFormer model can.</p>
</section>
"""

# Insert the explanation right after the opening header


# Add point accuracy section to HTML if computed
_pa_html = ""
try:
    _full_metrics_path = os.path.join(EXPERIMENT_DIR, "metrics_full.json")
    if os.path.exists(_full_metrics_path):
        with open(_full_metrics_path) as _f:
            _full = _json.load(_f)
        if "point_accuracy" in _full:
            _pa = _full["point_accuracy"]
            _delta = _pa["delta_vs_coralnet"]
            _color = "#2a8a3a" if _delta >= 0 else "#a83232"
            _verdict = f"BEATS CoralNet by {_delta*100:+.1f} pp" if _delta >= 0 else f"Below CoralNet by {abs(_delta)*100:.1f} pp"
            _pa_html = f"""
<section style='background:#fff;border:2px solid {_color};padding:20px;border-radius:8px;margin:24px 0;'>
  <h2 style='margin-top:0;color:{_color};'>🎯 Direct CoralNet comparison — point accuracy</h2>
  <p style='font-size:18px;margin:8px 0;'>
    <strong>This model: {_pa['overall']*100:.2f}%</strong>
    &nbsp;|&nbsp; CoralNet baseline: 72.00%
    &nbsp;|&nbsp; <strong style='color:{_color};'>{_verdict}</strong>
  </p>
  <p>Evaluated on {_pa['total']:,} CoralNet-style labeled points from <code>{os.path.basename(YOUR_CSV_PATH)}</code>.
     This is the same metric CoralNet reports — directly comparable.</p>
</section>"""
except Exception as _e:
    pass

_html_parts = [
    _pa_html,
    _metrics_explanation_html,
    "<!DOCTYPE html><html lang=\"en\"><head>",
    '<meta charset="UTF-8">',
    f"<title>SegFormer Coral Report — {EXPERIMENT_NAME}</title>",
    f"<style>{_css}</style></head><body>",
    f"<h1>SegFormer Coral Segmentation — Model Report</h1>",
    '<div class="meta">',
    f'<div><b>Experiment</b>{EXPERIMENT_NAME}</div>',
    f'<div><b>Training stage</b>{_stage_label}</div>',
    f'<div><b>Label merge</b>{_merge_note}</div>',
    f'<div><b>Classes evaluated</b>{_ncls} / {_ntot}</div>',
    f'<div><b>Generated</b>{datetime.now().strftime("%Y-%m-%d %H:%M")}</div>',
    "</div>",
    "<h2>Summary Metrics</h2>",
    '<div class="summary">',
    (f'<div class="metric"><div class="lbl">mIoU</div>'
     f'<div class="val">{_miou*100:.1f}%</div>'
     f'<div class="def">Mean Intersection-over-Union — the standard segmentation benchmark. '
     f'Averages over all classes equally (rare = common).</div></div>'),
    (f'<div class="metric"><div class="lbl">Pixel Accuracy</div>'
     f'<div class="val">{_pxacc*100:.1f}%</div>'
     f'<div class="def">% of pixels classified correctly. Dominated by common classes — '
     f'inflated when one class covers most of the image.</div></div>'),
    (f'<div class="metric"><div class="lbl">FWIoU</div>'
     f'<div class="val">{_fwiou*100:.1f}%</div>'
     f'<div class="def">Frequency-weighted IoU — common classes count more. '
     f'Realistic estimate of real-world performance.</div></div>'),
    (f'<div class="metric"><div class="lbl">Mean Dice / F1</div>'
     f'<div class="val">{_mdice*100:.1f}%</div>'
     f'<div class="def">Similar to mIoU but slightly more lenient. '
     f'Often reported in medical image segmentation.</div></div>'),
    (f'<div class="metric"><div class="lbl">Macro Recall</div>'
     f'<div class="val">{_mrec*100:.1f}%</div>'
     f'<div class="def">Of each class\u2019s true pixels, what fraction did the model find? '
     f'Averaged equally across classes.</div></div>'),
    (f'<div class="metric"><div class="lbl">Macro Precision</div>'
     f'<div class="val">{_mprec*100:.1f}%</div>'
     f'<div class="def">When the model predicts a class, how often is it right? '
     f'Low precision = model over-predicts.</div></div>'),
    "</div>",
    "<h2>Interpretation &amp; Guidance</h2>",
    '<p class="note">Note: CoralNet reports <em>point classification accuracy</em> '
    "(was each sparse annotation point correctly identified?). "
    "This model does <em>full image segmentation</em> (every pixel must be labelled). "
    "Full segmentation is harder, so mIoU numbers will be lower than CoralNet accuracy "
    "numbers — this is expected, not a weakness.</p>",
    _guidance_html,
    "<h2>Per-class Performance</h2>",
    '<p class="note">Green = IoU ≥ 65% (strong), yellow = 40–65% (moderate), '
    "red = &lt;40% (weak). Support = number of validation pixels.</p>",
    f'<img src="data:image/png;base64,{_bar_b64}" style="max-width:100%;margin:12px 0">',
    "<table><thead><tr>",
    "<th>Class</th><th>IoU</th><th>Dice</th><th>Precision</th><th>Recall</th>",
    "<th>Val pixels</th><th>IoU bar</th></tr></thead>",
    f"<tbody>{_table_rows_html}</tbody></table>",
    _cm_section,
    "<h2>Top Misclassifications</h2>",
    '<p class="note">Which classes does the model confuse most? '
    "Values &gt;20% (red) are meaningful weaknesses.</p>",
    "<table><thead><tr><th>True class</th><th></th><th>Predicted as</th>",
    "<th>Pixels</th><th>% of true class</th></tr></thead>",
    f"<tbody>{_conf_rows_html}</tbody></table>",
    "<h2>Metric Definitions</h2>",
    "<table><thead><tr><th>Metric</th><th>Formula</th><th>Plain meaning</th></tr></thead><tbody>",
    "<tr><td><b>IoU</b></td><td>TP/(TP+FP+FN)</td>"
    "<td>Overlap between predicted and true region. 1.0 = perfect.</td></tr>",
    "<tr><td><b>Dice/F1</b></td><td>2TP/(2TP+FP+FN)</td>"
    "<td>Similar to IoU, slightly more lenient on partial overlaps.</td></tr>",
    "<tr><td><b>Precision</b></td><td>TP/(TP+FP)</td>"
    "<td>Of pixels labelled as this class, how many were correct?</td></tr>",
    "<tr><td><b>Recall</b></td><td>TP/(TP+FN)</td>"
    "<td>Of pixels that truly are this class, how many did we find?</td></tr>",
    "<tr><td><b>mIoU</b></td><td>mean(IoU per class)</td>"
    "<td>Headline number. Rare and common classes weighted equally.</td></tr>",
    "<tr><td><b>FWIoU</b></td><td>Σ freq_c × IoU_c</td>"
    "<td>Like mIoU but weighted by how common each class is.</td></tr>",
    "<tr><td><b>Pixel acc.</b></td><td>Σ correct / Σ total</td>"
    "<td>Overall % correct pixels. Easy to inflate with dominant classes.</td></tr>",
    "</tbody></table>",
    f'<footer>Generated by SegFormer 2-head coral training notebook — '
    f'{datetime.now().strftime("%Y-%m-%d")}</footer>',
    "</body></html>",
]
_html = "\n".join(_html_parts)

# Save
_report_path = os.path.join(EXPERIMENT_DIR, "report_ewout.html")
with open(_report_path, "w", encoding="utf-8") as _f:
    _f.write(_html)

_drive_report = os.path.join(DRIVE_DIR, f"report_{EXPERIMENT_NAME}.html")
try:
    shutil.copy(_report_path, _drive_report)
    print(f"Report saved:")
    print(f"  {_report_path}")
    print(f"  {_drive_report}   <- download this from Drive and send to Ewout")
except Exception as _e:
    print(f"Report saved: {_report_path}  (Drive copy failed: {_e})")
print("Open report_ewout.html in a browser — fully self-contained, no internet needed.")


In [ ]:
# ===========================================================================
# CELL 10: Validation layer 3 — visualize predictions on N fixed test images
# ===========================================================================
# Includes the SAME seed images used by sam_viewer.py and batch_export_sam.ipynb,
# then pads up to N_TEST_IMAGES (Cell 2) with random val-set images so you
# always get a fuller picture. Random picks are seeded by SEED for stability.
#
# Each row shows:
#   [original] [prediction colored] [blend] [prediction with class names]
# Class names are drawn DIRECTLY on region centroids in large, readable text.
# ===========================================================================

@torch.no_grad()
def predict_full_image(img_rgb, dataset_id=1):
    """Resize → normalize → forward → crop-pad → upsample back to original size."""
    model.eval()
    h0, w0 = img_rgb.shape[:2]
    tf = A.Compose([
        A.LongestMaxSize(max_size=INPUT_SIZE),
        A.PadIfNeeded(INPUT_SIZE, INPUT_SIZE, border_mode=cv2.BORDER_CONSTANT,
                      fill=0, mask_fill_value=IGNORE_INDEX),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    x = tf(image=img_rgb)['image'].unsqueeze(0).to(device)
    with _autocast(USE_AMP):
        logits = model(x, dataset_id)
    pred = logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

    # Undo pad/resize back to original resolution
    scale = INPUT_SIZE / max(h0, w0)
    new_h, new_w = int(h0 * scale), int(w0 * scale)
    py, px = (INPUT_SIZE - new_h) // 2, (INPUT_SIZE - new_w) // 2
    pred = pred[py:py+new_h, px:px+new_w]
    return cv2.resize(pred, (w0, h0), interpolation=cv2.INTER_NEAREST)


def colorize_mask(mask, palette, ignore=IGNORE_INDEX):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for ci in np.unique(mask):
        if ci == ignore: continue
        rgb[mask == ci] = palette[ci]
    return rgb


def find_test_image(name, images_dir):
    for candidate in [name, name.replace('.JPG', '.jpg'), name.replace('.jpg', '.JPG')]:
        p = os.path.join(images_dir, candidate)
        if os.path.exists(p): return p
    return None


def _draw_class_labels(img_rgb, pred_mask, idx_to_class, palette,
                       min_region_px=1500, ignore=IGNORE_INDEX):
    """Overlay class names on each connected region centroid, big readable text.
    Only labels regions ≥ min_region_px (keeps visualization uncluttered)."""
    out = img_rgb.copy()
    h, w = pred_mask.shape
    # Scale text with image size; looks good on 1500px-wide images.
    font_scale    = max(0.8, min(w, h) / 1400.0)
    thickness_fg  = max(2, int(round(font_scale * 2)))
    thickness_bg  = thickness_fg + 3
    font          = cv2.FONT_HERSHEY_SIMPLEX

    for ci in np.unique(pred_mask):
        if ci == ignore or ci >= len(idx_to_class):
            continue
        binary = (pred_mask == ci).astype(np.uint8)
        n_comp, comp, stats, cents = cv2.connectedComponentsWithStats(binary, 8)
        cname = idx_to_class[ci]
        col = tuple(int(c) for c in palette[ci])
        for k in range(1, n_comp):  # 0 is background
            area = stats[k, cv2.CC_STAT_AREA]
            if area < min_region_px:
                continue
            cx, cy = int(cents[k][0]), int(cents[k][1])
            # Make sure the centroid is inside the component (for elongated regions)
            if comp[cy, cx] != k:
                ys, xs = np.where(comp == k)
                cy, cx = int(ys.mean()), int(xs.mean())
            text = cname
            (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness_fg)
            tx = max(5, min(w - tw - 5, cx - tw // 2))
            ty = max(th + 5, min(h - 5, cy + th // 2))
            # Black halo for contrast
            cv2.putText(out, text, (tx, ty), font, font_scale,
                        (0, 0, 0), thickness_bg, cv2.LINE_AA)
            # Class color (brightened) for the fill
            bright = tuple(min(255, int(c) + 60) for c in col)
            cv2.putText(out, text, (tx, ty), font, font_scale,
                        bright, thickness_fg, cv2.LINE_AA)
    return out


# --- Expand TEST_IMAGES up to N_TEST_IMAGES, padding from val set (seeded) ---
_seed_images = [n for n in TEST_IMAGES if find_test_image(n, YOUR_IMAGES_DIR)]
_missing_seed = [n for n in TEST_IMAGES if n not in _seed_images]
if _missing_seed:
    print(f'ℹ️ Seed test images not found on disk (skipped): {_missing_seed}')

_viz_images = list(_seed_images)
if val_imgs_B and len(_viz_images) < N_TEST_IMAGES:
    rng = np.random.default_rng(SEED)
    pool = [os.path.basename(p) for p in val_imgs_B if os.path.basename(p) not in _viz_images]
    rng.shuffle(pool)
    for name in pool:
        if len(_viz_images) >= N_TEST_IMAGES:
            break
        _viz_images.append(name)

print(f'Visualizing {len(_viz_images)} images '
      f'({len(_seed_images)} seed + {len(_viz_images) - len(_seed_images)} random from val)')

n_rows = len(_viz_images)
fig, axes = plt.subplots(n_rows, 4, figsize=(26, 6.5 * n_rows))
if n_rows == 1:
    axes = [axes]

for row, tname in enumerate(_viz_images):
    p = find_test_image(tname, YOUR_IMAGES_DIR)
    if p is None:
        print(f'⚠️ Test image not found: {tname}')
        for ax in axes[row]: ax.axis('off')
        continue
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    pred = predict_full_image(img, dataset_id=1)
    pred_rgb = colorize_mask(pred, PALETTE_B)
    labelled = _draw_class_labels(pred_rgb, pred, YOUR_IDX_TO_CLASS, PALETTE_B)
    blend = cv2.addWeighted(img, 0.55, pred_rgb, 0.45, 0)
    blend_labelled = _draw_class_labels(blend, pred, YOUR_IDX_TO_CLASS, PALETTE_B)

    axes[row][0].imshow(img); axes[row][0].axis('off')
    axes[row][0].set_title(f'{tname}\n{img.shape[1]}×{img.shape[0]}', fontsize=11)
    axes[row][1].imshow(labelled); axes[row][1].axis('off')
    axes[row][1].set_title('Prediction (Head B) + class labels', fontsize=11)
    axes[row][2].imshow(blend); axes[row][2].axis('off')
    axes[row][2].set_title('Blend', fontsize=11)
    axes[row][3].imshow(blend_labelled); axes[row][3].axis('off')
    axes[row][3].set_title('Blend + class labels', fontsize=11)

# Legend — only classes actually shown across all visualized images
present = set()
for tname in _viz_images:
    p = find_test_image(tname, YOUR_IMAGES_DIR)
    if p is None: continue
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    present.update(np.unique(predict_full_image(img, 1)).tolist())
present.discard(IGNORE_INDEX)
patches = [mpatches.Patch(color=[c / 255 for c in PALETTE_B[i]], label=YOUR_IDX_TO_CLASS[i])
           for i in sorted(present) if i < N_CLASSES_B]
if patches:
    fig.legend(handles=patches, loc='lower center', ncol=min(10, len(patches)),
               fontsize=10, bbox_to_anchor=(0.5, -0.005))
plt.suptitle(f'SegFormer {MODEL_NAME} — stage="{STAGE}" — {len(_viz_images)} images',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# CELL 11: Export self-contained bundle + summary + README  →  auto-download
# ===========================================================================
# Writes, all into {EXPERIMENT_DIR}/ :
#   bundle.pt      — weights + class names + full config (use this in the app)
#   summary.json   — machine-readable record of the run
#   README.md      — human-readable "what was this run?" overview
#
# The bundle is self-sufficient — no notebook, Drive, or CSV needed to run it
# with `segformer_predict_app.py`.
#
# STANDALONE USE (re-wrap an existing checkpoint without retraining):
# run Cells 1 → 2 → 4 → 13 only. This cell derives num_classes and class names
# directly from the checkpoint, so the slow data cells can be skipped.
# ===========================================================================

import shutil
import json
import torch
from datetime import datetime

# --- Find the best checkpoint for this experiment --------------------------
best_ckpt = os.path.join(EXPERIMENT_DIR, 'best.pt')
if not os.path.exists(best_ckpt):
    # Back-compat: look in legacy flat checkpoints dir
    legacy = os.path.join(CKPT_DIR, f'{STAGE}_best.pt')
    if os.path.exists(legacy):
        print(f'ⓘ Using legacy checkpoint {legacy} (not inside experiment folder).')
        best_ckpt = legacy

assert os.path.exists(best_ckpt), (
    f'No best checkpoint found. Looked in:\n'
    f'  {EXPERIMENT_DIR}/best.pt\n'
    f'  {CKPT_DIR}/{STAGE}_best.pt\n'
    f'Train first (Cells 3 → 8) or change STAGE / EXPERIMENT_NAME in Cell 2.'
)

raw = torch.load(best_ckpt, map_location='cpu')
state_dict = raw['model'] if 'model' in raw else raw

# --- Derive num_classes from the state dict (works without running Cell 7) -
def _infer_num_classes(sd, head_prefix):
    for k, v in sd.items():
        if k.startswith(head_prefix) and k.endswith('classifier.weight'):
            return int(v.shape[0])
    raise RuntimeError(f'Could not find {head_prefix}classifier.weight in checkpoint.')

num_classes_a = _infer_num_classes(state_dict, 'head_a.')
num_classes_b = _infer_num_classes(state_dict, 'head_b.')

# --- Class names: prefer checkpoint, fall back to Cell 4 YOUR_CLASSES ------
def _recover_classes(ckpt_key, fallback_name, expected_n):
    v = raw.get(ckpt_key)
    if v and len(v) == expected_n:
        return list(v)
    if fallback_name in dir() and len(globals()[fallback_name]) == expected_n:
        print(f'ℹ️ {ckpt_key} missing/mismatched in checkpoint — using {fallback_name} from this session.')
        return list(globals()[fallback_name])
    print(f'⚠️ Could not recover {ckpt_key} — generating placeholder names.')
    return [f'class_{i}' for i in range(expected_n)]

your_classes_out        = _recover_classes('your_classes',        'YOUR_CLASSES',        num_classes_b)
coralscapes_classes_out = _recover_classes('coralscapes_classes', 'CORALSCAPES_CLASSES', num_classes_a)

# --- Gather training hyperparameters ---------------------------------------
def _g(name, default=None):
    return globals().get(name, default)

training_config = {
    'experiment_name':    EXPERIMENT_NAME,
    'stage':              raw.get('stage', STAGE),
    'model_name':         MODEL_NAME,
    'input_size':         INPUT_SIZE,
    'ignore_index':       IGNORE_INDEX,
    'batch_size':         _g('BATCH_SIZE'),
    'num_epochs':         _g('NUM_EPOCHS'),
    'lr':                 _g('LR'),
    'weight_decay':       _g('WEIGHT_DECAY'),
    'seed':               _g('SEED'),
    'use_amp':            _g('USE_AMP'),
    'point_radius':       _g('POINT_RADIUS'),
    'drop_classes':       list(_g('DROP_CLASSES', []) or []),
    'keep_top_n_classes': _g('KEEP_TOP_N_CLASSES'),
    'use_pretrained_coral':     _g('USE_PRETRAINED_CORAL'),
    'use_coralscapes_dataset':  _g('USE_CORALSCAPES_DATASET'),
    'coralscapes_weight_start': _g('CORALSCAPES_WEIGHT_START'),
    'coralscapes_weight_end':   _g('CORALSCAPES_WEIGHT_END'),
    'num_classes_b':      num_classes_b,
    'num_classes_a':      num_classes_a,
    'resumed_from':       _resolve_resume_path() if '_resolve_resume_path' in dir() else None,
}

# --- If SAM stage, snapshot the pipeline_settings from sam_coco.json -------
sam_settings = None
if training_config['stage'] == 'sam_masks' and os.path.exists(YOUR_SAM_COCO_PATH):
    try:
        with open(YOUR_SAM_COCO_PATH) as f:
            _coco = json.load(f)
        sam_settings = _coco.get('info', {}).get('pipeline_settings')
        print(f'✅ Captured SAM pipeline settings from {YOUR_SAM_COCO_PATH}')
    except Exception as e:
        print(f'⚠️ Could not read SAM settings from {YOUR_SAM_COCO_PATH}: {e}')

# --- Training history ------------------------------------------------------
# Prefer the persisted history.json (survives session restarts); fall back to
# in-memory `history` if the file is missing.
history_out = []
history_path = os.path.join(EXPERIMENT_DIR, 'history.json')
if os.path.exists(history_path):
    try:
        with open(history_path) as f:
            history_out = json.load(f)
    except Exception:
        pass
if not history_out:
    history_out = _g('history', [])

# --- Build the bundle ------------------------------------------------------
bundle = {
    'model_state':   state_dict,
    'model_name':    MODEL_NAME,
    'input_size':    INPUT_SIZE,
    'ignore_index':  IGNORE_INDEX,
    'num_classes_a': num_classes_a,
    'num_classes_b': num_classes_b,
    'your_classes':        your_classes_out,
    'coralscapes_classes': coralscapes_classes_out,
    'stage':            training_config['stage'],
    'experiment_name':  EXPERIMENT_NAME,
    'best_miou':        float(raw.get('miou', -1)),
    'epoch':            int(raw.get('epoch', -1)),
    'training_config':  training_config,
    'sam_settings':     sam_settings,
    'history':          history_out,
    'created':          datetime.utcnow().isoformat() + 'Z',
    'bundle_version':   2,
}

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
bundle_path_drive = os.path.join(EXPERIMENT_DIR, 'bundle.pt')
torch.save(bundle, bundle_path_drive)
size_mb = os.path.getsize(bundle_path_drive) / 1e6

# --- summary.json (machine-readable) ---------------------------------------
summary = {k: v for k, v in bundle.items() if k != 'model_state'}
summary_path = os.path.join(EXPERIMENT_DIR, 'summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

# --- README.md (human-readable overview) -----------------------------------
def _fmt_hist_row(h):
    def _f(v, fmt='.4f', default='—'):
        if v is None: return default
        try: return format(float(v), fmt)
        except (TypeError, ValueError): return default
    return (f"| {h.get('epoch','?'):>3} | {_f(h.get('train_loss'))} "
            f"| {_f(h.get('val_miou_B'))} "
            f"| {_f(h.get('val_miou_A'), default='—'):>7} "
            f"| {_f(h.get('duration_s'), '.0f')}s |")

readme_lines = [
    f'# 🧪 {EXPERIMENT_NAME}',
    '',
    f'**Stage:** `{bundle["stage"]}`     **Best val mIoU_B:** `{bundle["best_miou"]:.4f}` (epoch {bundle["epoch"]})',
    f'**Created:** {bundle["created"]}     **Bundle size:** {size_mb:.1f} MB',
    '',
    '## Model & training',
    '',
    f'- Architecture: `{MODEL_NAME}` (2-head SegFormer)',
    f'- Input size: `{INPUT_SIZE}` × `{INPUT_SIZE}`',
    f'- Head A (Coralscapes): **{num_classes_a}** classes  '
      f'({"pretrained" if training_config["use_pretrained_coral"] else "scratch"}, '
      f'{"joint-trained" if training_config["use_coralscapes_dataset"] else "regularizer off"})',
    f'- Head B (your data):   **{num_classes_b}** classes',
    f'- Epochs: `{training_config["num_epochs"]}`     Batch size: `{training_config["batch_size"]}`     LR: `{training_config["lr"]}`',
    f'- Weight decay: `{training_config["weight_decay"]}`     Seed: `{training_config["seed"]}`     AMP: `{training_config["use_amp"]}`',
    f'- Dropped classes: `{training_config["drop_classes"] or "(none)"}`',
    f'- Top-N class filter: `{training_config["keep_top_n_classes"] or "ALL"}`',
    f'- Resumed from: `{training_config["resumed_from"] or "pretrained Coralscapes"}`',
    '',
]

if sam_settings:
    readme_lines += [
        '## SAM pipeline settings (from sam_coco.json)',
        '',
        f'- SAM model: `{sam_settings.get("sam_model")}`  (fp16: `{sam_settings.get("float16")}`)',
        f'- AMG grid: `{sam_settings.get("amg_points_per_side")}` points/side',
        f'- IoU threshold: `{sam_settings.get("amg_pred_iou_thresh")}`',
        f'- Stability threshold: `{sam_settings.get("amg_stability_thresh")}`',
        f'- Max mask coverage: `{sam_settings.get("max_mask_pct")}%`',
        f'- Overlap resolution: `{sam_settings.get("overlap_resolution")}`',
        f'- Label method: `{sam_settings.get("label_method")}`',
        f'- Images processed: `{sam_settings.get("total_images_processed")}`     '
          f'Annotations: `{sam_settings.get("total_annotations")}`',
        '',
    ]

if history_out:
    readme_lines += ['## Per-epoch history', '',
                     '| Epoch | Train loss | val mIoU_B | val mIoU_A | Duration |',
                     '|------:|-----------:|-----------:|-----------:|---------:|']
    for h in history_out:
        readme_lines.append(_fmt_hist_row(h))
    readme_lines.append('')

readme_lines += [
    '## Classes trained (Head B)',
    '',
    ', '.join(f'`{c}`' for c in your_classes_out),
    '',
    '## Files in this experiment',
    '',
    '- `bundle.pt`   — self-contained model + metadata (use in `segformer_predict_app.py`)',
    '- `best.pt`     — raw checkpoint with highest val mIoU_B (used for resuming)',
    '- `final.pt`    — last-epoch checkpoint',
    '- `summary.json` — machine-readable version of this README',
    '- `history.json` — per-epoch metrics',
    '- `README.md`   — this file',
]
readme_path = os.path.join(EXPERIMENT_DIR, 'README.md')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(readme_lines))

# --- Report ----------------------------------------------------------------
print(f'\n✅ Experiment folder: {EXPERIMENT_DIR}')
print(f'   bundle.pt     {size_mb:.1f} MB')
print(f'   summary.json  (machine-readable)')
print(f'   README.md     (human-readable — read this)')
print(f'\n   stage={bundle["stage"]}  best mIoU_B={bundle["best_miou"]:.4f}  epoch={bundle["epoch"]}')
print(f'   classes: {num_classes_b} Head B + {num_classes_a} Head A')
if sam_settings:
    print(f'   SAM: grid={sam_settings.get("amg_points_per_side")}  '
          f'IoU>{sam_settings.get("amg_pred_iou_thresh")}  '
          f'stab>{sam_settings.get("amg_stability_thresh")}  '
          f'maxmask={sam_settings.get("max_mask_pct")}%')

# Convergence hint
if history_out and len(history_out) >= 5:
    last5 = [h.get('val_miou_B') for h in history_out[-5:] if h.get('val_miou_B') is not None]
    if len(last5) >= 2:
        delta = last5[-1] - last5[0]
        trend = '📈 still improving — consider more epochs' if delta > 0.005 \
                else '📉 regressing — early stopping might help' if delta < -0.005 \
                else '✅ converged — epoch count is fine'
        print(f'   Convergence (last 5 epochs, ΔmIoU_B={delta:+.4f}): {trend}')

# --- Browser download ------------------------------------------------------
local_copy = f'/content/{EXPERIMENT_NAME}__bundle.pt'
shutil.copyfile(bundle_path_drive, local_copy)
try:
    from google.colab import files
    print(f'\n⬇️  Starting download of {local_copy} ...')
    files.download(local_copy)
    print(f'   (if it does not start, open the Files panel and grab {local_copy})')
    print(f'   README and summary stay on Drive at {EXPERIMENT_DIR}')
except ImportError:
    print(f'\nNot in Colab — bundle at {bundle_path_drive}')

In [ ]:
# ===========================================================================
# CELL 12: Compare two experiments side-by-side
# ===========================================================================
# Tells you whether a newer experiment is actually better than a previous one.
# Prints:
#   - full settings comparison (epochs, classes, SAM params, drop list, ...)
#   - validation mIoU_B of both, on the SAME val set (if both bundles load
#     into the current model architecture)
#   - side-by-side prediction on TEST_IMAGES[0] if the model + val loader
#     from earlier cells are alive in this session
#   - verdict: ✅ B wins / ❌ B regressed / ➖ tie
#
# NOTHING is modified — both bundles are read-only. Safe to re-run.
# ===========================================================================

import json as _json

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  ✏️  EDIT THIS — which two experiments to compare                        │
# └─────────────────────────────────────────────────────────────────────────┘
# Leave as "" (empty) for auto-pick: most recent points vs most recent sam_masks.
# Or set to a folder name under {OUTPUT_DIR}/experiments/.
EXPERIMENT_A_NAME = ""      # "before" — e.g. "points_top35_clean_20ep"
EXPERIMENT_B_NAME = ""      # "after"  — e.g. "sam_masks_top35_clean_30ep"

COMPARE_IMG_NAME = TEST_IMAGES[0] if TEST_IMAGES else None


def _auto_pick(stage_prefix: str) -> str | None:
    """Pick the most recently modified experiment folder starting with `stage_prefix_`."""
    if not os.path.isdir(EXPERIMENTS_DIR):
        return None
    candidates = []
    for d in os.listdir(EXPERIMENTS_DIR):
        full = os.path.join(EXPERIMENTS_DIR, d)
        if os.path.isdir(full) and d.startswith(stage_prefix + '_') and \
           os.path.exists(os.path.join(full, 'bundle.pt')):
            candidates.append((os.path.getmtime(full), d))
    candidates.sort(reverse=True)
    return candidates[0][1] if candidates else None


if not EXPERIMENT_A_NAME:
    EXPERIMENT_A_NAME = _auto_pick('points') or ''
if not EXPERIMENT_B_NAME:
    EXPERIMENT_B_NAME = _auto_pick('sam_masks') or _auto_pick('pseudo') or ''


def _load_bundle_meta(exp_name):
    if not exp_name:
        return None
    bundle_path = os.path.join(EXPERIMENTS_DIR, exp_name, 'bundle.pt')
    if not os.path.exists(bundle_path):
        return None
    b = torch.load(bundle_path, map_location='cpu')
    return {
        'name':              exp_name,
        'path':              bundle_path,
        'stage':             b.get('stage'),
        'best_miou':         float(b.get('best_miou', b.get('miou', -1))),
        'epoch':             int(b.get('epoch', -1)),
        'training_config':   b.get('training_config') or {},
        'sam_settings':      b.get('sam_settings'),
        'history':           b.get('history') or [],
        'num_classes_b':     b.get('num_classes_b'),
        'num_classes_a':     b.get('num_classes_a'),
        'bundle_version':    b.get('bundle_version', 1),
        'state_dict':        b.get('model_state') or b.get('model'),
        'your_classes':      b.get('your_classes'),
    }


bundles_meta = {
    'A (before)': _load_bundle_meta(EXPERIMENT_A_NAME),
    'B (after)':  _load_bundle_meta(EXPERIMENT_B_NAME),
}

# ── Section 1: settings side-by-side ──────────────────────────────────────
print('═' * 88)
print(f'{"":>28}  {"A (before)":^26}  {"B (after)":^26}')
print(f'{"experiment":>28}  '
      f'{EXPERIMENT_A_NAME[:26]:^26}  '
      f'{EXPERIMENT_B_NAME[:26]:^26}')
print('─' * 88)
for label in ('stage', 'best_miou', 'epoch', 'num_classes_b', 'bundle_version'):
    vals = []
    for key in ('A (before)', 'B (after)'):
        m = bundles_meta[key]
        if m is None:           vals.append('—')
        elif label == 'best_miou': vals.append(f'{m[label]:.4f}')
        else:                    vals.append(str(m[label]))
    print(f'{label:>28}  {vals[0]:^26}  {vals[1]:^26}')

print('─' * 88 + '\nTraining config:')
_keys = ['num_epochs', 'batch_size', 'lr', 'weight_decay', 'input_size',
         'keep_top_n_classes', 'drop_classes', 'point_radius',
         'coralscapes_weight_start', 'coralscapes_weight_end',
         'use_pretrained_coral', 'use_coralscapes_dataset', 'resumed_from']
for k in _keys:
    vals = []
    for key in ('A (before)', 'B (after)'):
        m = bundles_meta[key]
        v = (m['training_config'].get(k) if m else None)
        vals.append(str(v) if v not in (None, []) else '—')
    print(f'  {k:>26}  {vals[0]:^26}  {vals[1]:^26}')

print('─' * 88 + '\nSAM pipeline settings (sam_masks stage only):')
_sam_keys = ['sam_model', 'amg_points_per_side', 'amg_pred_iou_thresh',
             'amg_stability_thresh', 'max_mask_pct',
             'total_images_processed', 'total_annotations']
for k in _sam_keys:
    vals = []
    for key in ('A (before)', 'B (after)'):
        m = bundles_meta[key]
        s = (m['sam_settings'] if m and m['sam_settings'] else {}) or {}
        vals.append(str(s.get(k, '—')))
    print(f'  {k:>26}  {vals[0]:^26}  {vals[1]:^26}')
print('═' * 88)


# ── Section 2: re-run validation on shared val set ────────────────────────
_have_both = all(m is not None for m in bundles_meta.values())
_have_val  = 'val_loader_B' in dir() and 'model' in dir() and 'evaluate' in dir()

if not _have_both:
    missing = [k for k, m in bundles_meta.items() if m is None]
    print(f'\nⓘ Skipping re-eval — missing bundles: {missing}')
    if not EXPERIMENT_A_NAME: print('  (set EXPERIMENT_A_NAME above)')
    if not EXPERIMENT_B_NAME: print('  (set EXPERIMENT_B_NAME above)')
elif not _have_val:
    print('\nⓘ Skipping re-eval — run Cells 6 + 7 + 8 first to build val_loader_B '
          'and model, or just compare settings above.')
else:
    # Only eval if class counts match the current model (otherwise load_state_dict
    # would silently reinit half the weights and skew the metric).
    def _compatible_with_current_model(bundle):
        cur = model.state_dict()
        for k, v in (bundle['state_dict'] or {}).items():
            if k in cur and cur[k].shape != v.shape:
                return False
        return True

    print('\nRe-running validation with both bundles on the same val set...')
    _baseline_state = {k: v.clone() for k, v in model.state_dict().items()}

    scores = {}
    for key, m in bundles_meta.items():
        if not _compatible_with_current_model(m):
            print(f'  ⚠️ {key} has incompatible class counts vs current model — skipping.')
            continue
        print(f'  Evaluating {key} ...')
        model.load_state_dict(m['state_dict'], strict=False)
        scores[key] = evaluate(model, val_loader_B, 1, N_CLASSES_B)

    model.load_state_dict(_baseline_state, strict=False)

    if len(scores) == 2:
        a, b = scores['A (before)']['miou'], scores['B (after)']['miou']
        print(f'\nValidation mIoU_B (higher is better):')
        print(f'  A (before): {a:.4f}')
        print(f'  B (after):  {b:.4f}')
        delta = b - a
        verdict = ('✅ B WINS — ship the new model'        if delta > 0.005 else
                   '❌ B REGRESSED — keep A, investigate'   if delta < -0.005 else
                   '➖ TIE — within noise. Spot-check visuals before deciding.')
        print(f'  Δ = {delta:+.4f}  →  {verdict}')


# ── Section 3: visual side-by-side ────────────────────────────────────────
if _have_both and COMPARE_IMG_NAME and 'predict_full_image' in dir():
    _img_path = os.path.join(YOUR_IMAGES_DIR, COMPARE_IMG_NAME)
    if os.path.exists(_img_path):
        print(f'\nRendering side-by-side on {COMPARE_IMG_NAME}...')
        _img = cv2.cvtColor(cv2.imread(_img_path), cv2.COLOR_BGR2RGB)
        _baseline = {k: v.clone() for k, v in model.state_dict().items()}

        fig, axs = plt.subplots(1, 3, figsize=(22, 7))
        axs[0].imshow(_img); axs[0].set_title(COMPARE_IMG_NAME, fontsize=11); axs[0].axis('off')
        for i, (key, suffix) in enumerate([('A (before)', 'A'), ('B (after)', 'B')]):
            m = bundles_meta[key]
            try:
                model.load_state_dict(m['state_dict'], strict=False)
                pred = predict_full_image(_img, dataset_id=1)
                pred_rgb = colorize_mask(pred, PALETTE_B)
                blend = cv2.addWeighted(_img, 0.55, pred_rgb, 0.45, 0)
                axs[i + 1].imshow(blend)
                axs[i + 1].set_title(f'{suffix}: {m["name"]}', fontsize=11, fontweight='bold')
                axs[i + 1].axis('off')
            except Exception as e:
                axs[i + 1].set_title(f'{suffix}: failed ({e})', fontsize=10, color='red')
                axs[i + 1].axis('off')
        model.load_state_dict(_baseline, strict=False)
        plt.suptitle('Before vs After — same image, same model architecture',
                      fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

In [ ]:
# ===========================================================================
# CELL 13: (Optional) Generate pseudo-labels for the next stage
# ===========================================================================
# After training a stage, the model has opinions about UNLABELED pixels too.
# If those opinions are very confident (softmax > conf_thresh), we can treat
# them as labels for the next training round. Uncertain pixels stay as 255.
#
# This expands your effective training coverage without extra annotation work.
# Use sparingly (1-2 rounds) — accumulated errors compound.
#
# USAGE: uncomment the call below to generate masks in masks_cache/pseudo/,
# then set STAGE = "pseudo" in Cell 2 and re-run Cells 2 → 11.
# ===========================================================================

PSEUDO_CONF_THRESH = 0.90      # only keep predictions with softmax > this
PSEUDO_OUTPUT_DIR  = os.path.join(MASKS_CACHE_DIR, 'pseudo')


def generate_pseudo_labels(images_dir, masks_dir_base, out_dir, dataset_id=1,
                           conf_thresh=PSEUDO_CONF_THRESH):
    """
    For each image in images_dir:
      - run model, get per-pixel softmax prediction
      - start with the existing mask (if any); keep its labeled pixels as-is
      - for pixels that were IGNORE and now have confidence > conf_thresh,
        fill in the predicted class
      - everything else stays IGNORE
    """
    os.makedirs(out_dir, exist_ok=True)
    model.eval()
    tf = A.Compose([
        A.LongestMaxSize(max_size=INPUT_SIZE),
        A.PadIfNeeded(INPUT_SIZE, INPUT_SIZE, border_mode=cv2.BORDER_CONSTANT,
                      fill=0, mask_fill_value=IGNORE_INDEX),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    exts = {'.jpg', '.jpeg', '.png'}
    files = [f for f in os.listdir(images_dir)
             if os.path.splitext(f)[1].lower() in exts]
    n = 0

    for fname in files:
        img = cv2.cvtColor(cv2.imread(os.path.join(images_dir, fname)), cv2.COLOR_BGR2RGB)
        h0, w0 = img.shape[:2]
        x = tf(image=img)['image'].unsqueeze(0).to(device)
        with torch.no_grad(), _autocast(USE_AMP):
            logits = model(x, dataset_id)
        prob = F.softmax(logits, dim=1)[0]
        conf, pred = prob.max(0)
        conf = conf.cpu().numpy(); pred = pred.cpu().numpy().astype(np.uint8)

        # Undo pad/resize
        scale = INPUT_SIZE / max(h0, w0)
        new_h, new_w = int(h0 * scale), int(w0 * scale)
        py, px = (INPUT_SIZE - new_h) // 2, (INPUT_SIZE - new_w) // 2
        conf = cv2.resize(conf[py:py+new_h, px:px+new_w], (w0, h0), interpolation=cv2.INTER_LINEAR)
        pred = cv2.resize(pred[py:py+new_h, px:px+new_w], (w0, h0), interpolation=cv2.INTER_NEAREST)

        # Start from existing mask (preserve human/SAM labels), fill confident gaps
        base_path = os.path.join(masks_dir_base, os.path.splitext(fname)[0] + '.png') \
            if masks_dir_base else None
        if base_path and os.path.exists(base_path):
            out_mask = cv2.imread(base_path, 0)
        else:
            out_mask = np.full((h0, w0), IGNORE_INDEX, dtype=np.uint8)

        fill = (out_mask == IGNORE_INDEX) & (conf >= conf_thresh)
        out_mask[fill] = pred[fill]
        cv2.imwrite(os.path.join(out_dir, os.path.splitext(fname)[0] + '.png'), out_mask)
        n += 1
        if n % 200 == 0: print(f'  {n} images pseudo-labeled')

    print(f'✅ Pseudo-labels written for {n} images → {out_dir}')


# --- Uncomment to run (~10-15 min for ~4600 images on L4) ---
# generate_pseudo_labels(
#     YOUR_IMAGES_DIR,
#     stage_mask_dir,    # previous stage's masks — their labels are preserved
#     PSEUDO_OUTPUT_DIR,
#     dataset_id=1,
#     conf_thresh=PSEUDO_CONF_THRESH,
# )
# print('Next: set STAGE = "pseudo" in Cell 2 and re-run Cells 2 → 11')

print('Cell 13: pseudo-label generator defined. Uncomment the call above to run it.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — TRACK RECORD: scan all experiments, build comparison overview
# Updates runs_overview.md in EXPERIMENTS_DIR each time you run it.
# Shows settings, mIoU, point accuracy, classes, top/bottom classes per run.
# ═══════════════════════════════════════════════════════════════════════════
import os, json
from datetime import datetime

def _safe_load_json(p):
    try:
        with open(p, encoding="utf-8") as f: return json.load(f)
    except Exception: return None

def _safe_load_pt(p):
    try:
        return torch.load(p, map_location="cpu", weights_only=False)
    except Exception: return None

def _scan_experiments(root):
    runs = []
    if not os.path.isdir(root): return runs
    for name in sorted(os.listdir(root)):
        d = os.path.join(root, name)
        if not os.path.isdir(d): continue
        rec = {"name": name, "dir": d}

        best_pt = os.path.join(d, "best.pt")
        if os.path.exists(best_pt):
            sd = _safe_load_pt(best_pt)
            if sd:
                rec["best_miou"]  = float(sd.get("miou", -1))
                rec["best_epoch"] = int(sd.get("epoch", -1))
                rec["stage"]      = sd.get("stage", "?")
            rec["ckpt_size_mb"] = round(os.path.getsize(best_pt)/1e6, 1)
            rec["mtime"]        = datetime.fromtimestamp(os.path.getmtime(best_pt)).strftime("%Y-%m-%d %H:%M")
        else:
            rec["best_miou"] = None

        mf = _safe_load_json(os.path.join(d, "metrics_full.json"))
        if mf:
            summary = mf.get("summary", {})
            rec["mIoU"]           = summary.get("mIoU")
            rec["FWIoU"]          = summary.get("FWIoU")
            rec["pixel_acc"]      = summary.get("pixel_accuracy")
            rec["mDice"]          = summary.get("mDice_F1")
            rec["n_classes"]      = summary.get("n_classes_total")
            rec["n_with_support"] = summary.get("n_classes_with_support")

            pc = mf.get("per_class", {})
            scored = [(cls, m["iou"]) for cls, m in pc.items()
                      if m.get("iou") is not None and m.get("support", 0) > 100]
            scored.sort(key=lambda x: -x[1])
            rec["top5_classes"]    = scored[:5]
            rec["bottom3_classes"] = scored[-3:] if len(scored) >= 3 else []

            pa = mf.get("point_accuracy")
            if pa:
                rec["point_acc"]         = pa.get("overall")
                rec["point_correct"]     = pa.get("correct")
                rec["point_total"]       = pa.get("total")
                rec["delta_vs_coralnet"] = pa.get("delta_vs_coralnet")

        sj = _safe_load_json(os.path.join(d, "summary.json"))
        if sj:
            rec["epochs"]       = sj.get("epochs") or sj.get("n_epochs")
            rec["lr"]           = sj.get("lr")
            rec["batch_size"]   = sj.get("batch_size")
            rec["input_size"]   = sj.get("input_size")
            rec["keep_top_n"]   = sj.get("keep_top_n_classes")
            rec["use_merge"]    = sj.get("use_label_merge")
            rec["point_radius"] = sj.get("point_radius")

        hj = _safe_load_json(os.path.join(d, "history.json"))
        if hj and isinstance(hj, list) and hj:
            rec["n_epochs_run"] = len(hj)
            rec.setdefault("epochs", len(hj))

        runs.append(rec)

    runs.sort(key=lambda r: (r.get("best_miou") or -1), reverse=True)
    return runs

def _fmt(v, fmt=".4f", default="—"):
    if v is None: return default
    try: return format(float(v), fmt)
    except (TypeError, ValueError): return str(v)

def _fmt_pct(v, default="—"):
    return f"{v*100:.1f}%" if v is not None else default

# ── Run scan ─────────────────────────────────────────────────────────────────
runs = _scan_experiments(EXPERIMENTS_DIR)

if not runs:
    print(f"No experiments found at {EXPERIMENTS_DIR}")
else:
    print("═" * 110)
    print(f"  📊 ALL RUNS  —  {len(runs)} experiments in {EXPERIMENTS_DIR}")
    print("═" * 110)
    _h = f'  {"Rank":<5}{"Name":<42}{"mIoU":>8}{"PointAcc":>10}{"vs72%":>9}{"Stage":>12}{"Date":>17}'
    print(_h)
    print("  " + "-" * 105)
    for i, r in enumerate(runs, 1):
        miou_s   = _fmt(r.get("best_miou"))
        pa       = r.get("point_acc")
        pa_s     = _fmt_pct(pa)
        delta    = r.get("delta_vs_coralnet")
        if delta is None:
            delta_s = "—"
        elif delta >= 0:
            delta_s = f"+{delta*100:.1f}pp"
        else:
            delta_s = f"{delta*100:.1f}pp"
        stage_s  = (r.get("stage") or "?")[:11]
        date_s   = r.get("mtime", "—")
        marker   = "🏆 " if i == 1 else "   "
        nm       = r["name"]
        print(f"  {marker}{i:<2}{nm:<42}{miou_s:>8}{pa_s:>10}{delta_s:>9}{stage_s:>12}{date_s:>17}")

    best = runs[0]
    print()
    print("═" * 110)
    print(f"  🏆 BEST RUN: {best['name']}")
    print("═" * 110)
    print(f"  mIoU:               {_fmt(best.get('best_miou'))}")
    print(f"  Pixel accuracy:     {_fmt_pct(best.get('pixel_acc'))}")
    print(f"  FWIoU:              {_fmt(best.get('FWIoU'))}")
    if best.get("point_acc") is not None:
        pc = best.get('point_correct', 0)
        pt = best.get('point_total', 0)
        print(f"  Point accuracy:     {_fmt_pct(best['point_acc'])}  ({pc:,}/{pt:,})")
        d = best.get("delta_vs_coralnet")
        if d is not None:
            verdict = "BEATS" if d >= 0 else "BELOW"
            print(f"  vs CoralNet 72%:    {verdict} by {abs(d)*100:.1f}pp")
    if best.get("top5_classes"):
        print(f"  Top 5 classes:")
        for cls, iou in best["top5_classes"]:
            print(f"    {cls:<20} IoU={iou:.3f}")
    if best.get("bottom3_classes"):
        print(f"  Bottom 3 classes:")
        for cls, iou in best["bottom3_classes"]:
            print(f"    {cls:<20} IoU={iou:.3f}")
    print(f"  Classes (with support / total): {best.get('n_with_support','?')}/{best.get('n_classes','?')}")

    md_path = os.path.join(EXPERIMENTS_DIR, "runs_overview.md")
    md = ["# 📊 SegFormer Coral — Runs Overview", ""]
    md.append(f"_Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}_   _Total runs: {len(runs)}_")
    md += ["", "## Comparison table", ""]
    md.append("| Rank | Run name | mIoU | Point acc | vs 72% | Pixel acc | Classes (active/total) | Stage | Epoch | Date |")
    md.append("|-----:|----------|-----:|----------:|-------:|----------:|----------------------:|:------|------:|:-----|")
    for i, r in enumerate(runs, 1):
        d = r.get("delta_vs_coralnet")
        if d is None:
            delta_s = "—"
        elif d >= 0:
            delta_s = f"+{d*100:.1f}pp"
        else:
            delta_s = f"{d*100:.1f}pp"
        rank_s = "🏆 "+str(i) if i == 1 else str(i)
        n_active = r.get("n_with_support", "?")
        n_total = r.get("n_classes", "?")
        md.append(f"| {rank_s} | `{r['name']}` | {_fmt(r.get('best_miou'))} | {_fmt_pct(r.get('point_acc'))} | {delta_s} | {_fmt_pct(r.get('pixel_acc'))} | {n_active}/{n_total} | {r.get('stage','?')} | {r.get('best_epoch','?')} | {r.get('mtime','—')} |")

    md += ["", "## Per-run details", ""]
    for r in runs:
        md.append(f"### `{r['name']}`")
        md.append("")
        md.append(f"- **Best mIoU:** {_fmt(r.get('best_miou'))} at epoch {r.get('best_epoch','?')}")
        if r.get("point_acc") is not None:
            pc_v = r.get('point_correct', 0)
            pt_v = r.get('point_total', 0)
            md.append(f"- **Point accuracy:** {_fmt_pct(r['point_acc'])} ({pc_v:,}/{pt_v:,})")
        md.append(f"- **Stage:** `{r.get('stage','?')}`")
        md.append(f"- **Pixel accuracy:** {_fmt_pct(r.get('pixel_acc'))}")
        md.append(f"- **FWIoU:** {_fmt(r.get('FWIoU'))}")
        md.append(f"- **Classes (active/total):** {r.get('n_with_support','?')}/{r.get('n_classes','?')}")
        md.append(f"- **Trained:** {r.get('mtime','—')}")
        md.append(f"- **Checkpoint size:** {r.get('ckpt_size_mb','?')} MB")
        if r.get("top5_classes"):
            md.append("- **Top 5 classes (by IoU):**")
            for cls, iou in r["top5_classes"]:
                md.append(f"  - `{cls}` — IoU {iou:.3f}")
        if r.get("bottom3_classes"):
            md.append("- **Bottom 3 classes (by IoU):**")
            for cls, iou in r["bottom3_classes"]:
                md.append(f"  - `{cls}` — IoU {iou:.3f}")
        md.append("")

    yc = globals().get("YOUR_CLASSES", [])
    md += ["## Current notebook session settings", ""]
    md.append(f"- `TRAINING_FLOW`: `{globals().get('TRAINING_FLOW', '?')}`")
    md.append(f"- `MODEL_NAME`: `{globals().get('MODEL_NAME', '?')}`")
    md.append(f"- `INPUT_SIZE`: {globals().get('INPUT_SIZE', '?')}")
    md.append(f"- `BATCH_SIZE`: {globals().get('BATCH_SIZE', '?')}")
    md.append(f"- `KEEP_TOP_N_CLASSES`: {globals().get('KEEP_TOP_N_CLASSES', '?')}")
    md.append(f"- `USE_LABEL_MERGE`: {globals().get('USE_LABEL_MERGE', '?')}")
    md.append(f"- `POINT_RADIUS`: {globals().get('POINT_RADIUS', '?')}")
    md.append(f"- `RESUME_FROM_EXPERIMENT`: `{globals().get('RESUME_FROM_EXPERIMENT', None)}`")
    _yc_extra = f", ... +{len(yc)-20} more" if len(yc) > 20 else ""
    md.append(f"- `YOUR_CLASSES` ({len(yc)}): {', '.join(yc[:20])}{_yc_extra}")
    md.append("")

    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md))

    print()
    print(f"  💾 Overview saved to {md_path}")
    print(f"     Open in any markdown viewer for the full per-run breakdown.")


---

# 🎯 What to do next

## 1. Decide if this run is good enough
- Check the last lines of **Cell 11** output:
  - `📈 still improving` → bump `NUM_EPOCHS` in Cell 2, re-run.
  - `✅ converged` → you're done with this experiment.
  - `📉 regressing` → something is off; inspect `history.json` and consider early stopping.
- Open `{EXPERIMENT_DIR}/README.md` on Drive — it has everything in one place.

## 2. Validate visually
- Run the Streamlit app: `streamlit run webapp/segformer_predict_app.py`
- Upload `{EXPERIMENT_DIR}/bundle.pt` (or use the one that just auto-downloaded).
- Click through the 10 quick-load test images. Do the masks look like the image content?

## 3. Move to the next stage

| You just finished | Do this next |
|---|---|
| `points` (first run) | Finish `batch_export_sam.ipynb` → you get `sam_coco.json`. Then set `STAGE = "sam_masks"`, `NUM_EPOCHS = 30`, re-run this notebook. |
| `sam_masks` | Run **Cell 12** to compare against `points`. If B wins by ≥ 0.05 mIoU, ship it. Otherwise tune SAM params and re-export. |
| Good `sam_masks` | Optional: try `STAGE = "pseudo"` — only helps if sam_masks clearly converged. |

## 4. Things worth experimenting with (in order of expected payoff)

- **Top-N cap**: try `KEEP_TOP_N_CLASSES = 35`, `50`, `None`. Smaller N → higher mIoU on the classes that matter, lower tail coverage.
- **Drop list**: audit your CSV for sentinel labels (Unknown, Off, NA) and add any you find to `DROP_CLASSES`.
- **Longer sam_masks**: 30 epochs is usually enough, but try 50 on a converged baseline.
- **SAM grid density**: in `batch_export_sam.ipynb` Cell 2, try `AMG_POINTS_PER_SIDE = 16` (slower but catches smaller colonies). Re-export, then compare the two `sam_masks_*` experiments with Cell 12.
- **Joint Coralscapes training**: `USE_CORALSCAPES_DATASET = True` — rarely helps on top of the pretrained init, but worth testing if you're stuck.

## 5. When to stop iterating

When the **visual check in the Streamlit app** looks good on your hardest images AND the dominant-class IoU in `README.md` is > 0.60. Pixel mIoU will cap around 0.50 on this kind of data regardless of technique — don't chase numbers past the visual ceiling.

---

### 🔍 Still to investigate

- Can we skip the 10% largest AMG masks during SAM export? They're usually sand/water and dilute training.
- Is there a systematic class confusion (e.g. Por ↔ Acr)? Run `detailed_eval` (Cell 9) on both models and diff the confusion matrices.
- Try `MODEL_NAME = "nvidia/mit-b3"` if training is cheap enough — bigger model, ~2× params, +2-4 mIoU typically.